# KLTN EpiWeather ML — Pipeline v3

**Sinh viên:** Phạm Hữu Luân | MSSV: 110122016 | Lớp: DA22TTA

**Đề tài:** Hệ thống cảnh báo nguy cơ dịch bệnh theo mùa dựa trên dữ liệu y tế và thời tiết toàn cầu

---

## Approach (chốt 13/05/2026) — Ordinal Multi-class Classification với Endemic Channel labels

```
TẦNG 1 — Endemic Channel Label (off-line, không học)
  per (iso3, week_of_year): baseline = mean(5 năm same week) ± 2σ
  label: Low / Medium / High
TẦNG 2 — Multi-model comparison → winner
  Naive / Logistic / RandomForest / XGBClassifier / LightGBM
  Walk-forward CV (val_year 2014–2019, 6 folds)
Output: P(Low), P(Medium), P(High) per (country, week, disease)
```

---

## Pipeline

| # | SESSION | Input → Output |
|---|---|---|
| 0 | Setup | — → môi trường Colab |
| 1 | Load raw | Drive files → DataFrames |
| 2 | Data Quality | DataFrames → missing/coverage report |
| 3 | EDA (raw) | flu/dengue raw → seasonality + heatmap |
| 4 | ERA5 | NetCDF → era5_weekly.csv *(idempotent)* |
| 5 | Merge | flu + dengue + ERA5 → master_weekly.csv *(idempotent)* |
| **6** | **EDA mở rộng (8 bước)** | **master_weekly → data OK / flag issues** |
| 7 | CCF Analysis | master_weekly → lag tối ưu per disease |
| 8 | Feature Engineering | master_weekly → features_flu.csv, features_dengue.csv |
| 9 | Endemic Channel Labels | features → risk_label (Low/Med/High) |
| 10 | Train Multi-model | features + labels → 5 models |
| 11 | So sánh model | CV scores → winner |
| 12 | Eval + Export | winner → metrics.json + .pkl |


# ⚙️ SESSION 0 — SETUP & RESTART CELL
> Chạy toàn bộ session này sau mỗi lần restart Colab runtime.
> Các session sau đọc data từ CSV — không cần chạy lại session nặng.

In [ ]:
# [0.1] Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive/KLTN')
if BASE.exists():
    print(f"✅ Drive mounted, BASE exists: {BASE}")
else:
    print(f"⚠️ BASE không tồn tại: {BASE} — kiểm tra lại cấu trúc thư mục Drive")

📌 **[0.1]** Mount Google Drive để truy cập data. `force_remount=False` giúp tránh unmount 
nếu Drive đã được mount từ trước (ví dụ khi chạy lại cell). Nếu BASE không tồn tại, 
kiểm tra lại tên thư mục trên Drive — phân biệt hoa/thường.

In [ ]:
# [0.2] Install missing libraries
import importlib, subprocess, sys

def ensure_package(import_name, pip_name=None):
    """Chi pip install neu package chua import duoc."""
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f'✅ {import_name} already installed')
    except ImportError:
        print(f'📦 Installing {pip_name}...')
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])
        print(f'✅ {pip_name} installed')

ensure_package('xgboost')
ensure_package('prophet')
ensure_package('scipy')
ensure_package('sklearn', 'scikit-learn')
ensure_package('joblib')
print("\n✅ Tat ca packages san sang")

📌 **[0.2]** Kiểm tra package trước khi install tránh re-install mất thời gian (~2-3 phút mỗi lần). 
Google Colab đã có sẵn `scikit-learn`, `scipy`, `xgboost` — thường chỉ cần install `prophet`. 
`prophet` yêu cầu `pystan` và compile C++ nên lần đầu cài sẽ lâu hơn.

In [ ]:
# [0.3] Import tat ca thu vien
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import json
from pathlib import Path
from typing import List, Dict, Tuple, Optional

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✅ Tat ca imports thanh cong")
print(f"  pandas {pd.__version__} | numpy {np.__version__}")

📌 **[0.3]** Tập trung tất cả import vào một cell duy nhất. Khi Colab restart, chỉ cần chạy 
lại Session 0 (5 cells) là đủ — không cần tìm import rải rác trong các session khác. 
`warnings.filterwarnings("ignore")` ẩn các DeprecationWarning từ Prophet và XGBoost để output dễ đọc hơn.

In [ ]:
# [0.4] Paths + Constants

BASE         = Path('/content/drive/MyDrive/KLTN')
RAW          = BASE / 'dataset' / 'epidemic' / 'raw'
PROCESSED    = BASE / 'dataset' / 'processed'
WEATHER_DIR  = BASE / 'dataset' / 'weather' / 'processed'
MODELS_DIR   = BASE / 'models'
OUTPUTS_DIR  = BASE / 'outputs'
FEATURES_DIR = PROCESSED

ERA5_FILE    = WEATHER_DIR / 'era5_weekly_2010_2019_final.csv'
MASTER_FILE  = PROCESSED   / 'master_weekly_2010_2019.csv'
FLUNET_FILE  = RAW          / 'VIW_FNT.csv'
DENGUE_FILE  = RAW          / 'National_extract_V1_3.csv'
ECDC_ILI     = RAW          / 'ILIARIRates.csv'
ECDC_SENT    = RAW          / 'sentinelTestsDetectionsPositivity.csv'

FEATURES_FLU_FILE    = FEATURES_DIR / 'features_flu_2010_2019.csv'
FEATURES_DENGUE_FILE = FEATURES_DIR / 'features_dengue_2010_2019.csv'
MODEL_FLU_FILE       = MODELS_DIR   / 'xgb_flu_final.pkl'
MODEL_DENGUE_FILE    = MODELS_DIR   / 'xgb_dengue_final.pkl'
FEATURE_LIST_FILE    = OUTPUTS_DIR  / 'feature_list.json'

for d in [MODELS_DIR, OUTPUTS_DIR, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_START  = 2010
TRAIN_END    = 2019
VAL_YEAR     = 2022
COVID_YEARS  = [2020, 2021]
TARGET_FLU    = 'inf_cases'
TARGET_DENGUE = 'dengue_total'
LAG_FLU    = [1, 2, 3]
LAG_DENGUE = [6, 8, 10, 12, 14]
WEATHER_VARS = [
    'temp_c', 'dewpoint_c', 'temp_min_c', 'temp_max_c', 'temp_range_c',
    'humidity_pct', 'wind_ms', 'precip_mm', 'conv_precip_mm', 'ls_precip_mm',
    'evap_mm', 'water_vapour', 'solar_wm2', 'uv_wm2', 'thermal_wm2',
    'cloud_cover', 'msl_pa', 'blh_m'
]

print("✅ Paths va constants da khai bao")
print(f"  TRAIN: {TRAIN_START}–{TRAIN_END} | VAL: {VAL_YEAR}")
print(f"  Weather vars: {len(WEATHER_VARS)} bien")
FILES = {
    'flunet'   : RAW / 'VIW_FNT.csv',
    'flu_meta' : RAW / 'VIW_FLU_METADATA.csv',
    'dengue'   : RAW / 'National_extract_V1_3.csv',
    'ecdc_sen' : RAW / 'sentinelTestsDetectionsPositivity.csv',
    'ecdc_ili' : RAW / 'ILIARIRates.csv',
    'era5'     : ERA5_FILE,
    'master'   : MASTER_FILE,
}
print('FILES:', {k: v.name for k, v in FILES.items()})

📌 **[0.4]** Tất cả path và constant khai báo tại đây — các session sau chỉ cần chạy lại 
Session 0 để có đủ biến, không tự khai báo lại tránh inconsistency. 
`mkdir(parents=True, exist_ok=True)` đảm bảo idempotent — chạy lại không báo lỗi dù thư mục đã tồn tại.

In [ ]:
# [0.5] File verification

files_to_check = {
    "MASTER_FILE":  MASTER_FILE,
    "ERA5_FILE":    ERA5_FILE,
    "FLUNET_FILE":  FLUNET_FILE,
    "DENGUE_FILE":  DENGUE_FILE,
    "ECDC_ILI":     ECDC_ILI,
    "ECDC_SENT":    ECDC_SENT,
}

for name, path in files_to_check.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f'✅ {name}: {path.name} ({size_mb:.1f} MB)')
    else:
        print(f'⚠️  {name}: KHONG TIM THAY -> {path}')

if MASTER_FILE.exists():
    _m = pd.read_csv(MASTER_FILE)
    print(f"\n📊 master_weekly shape: {_m.shape}")
    print(f"   Columns: {list(_m.columns)}")
    print(f"   iso3 unique: {_m[\"iso3\"].nunique()} countries")
    print(f"   year range: {_m[\"iso_year\"].min()}–{_m[\"iso_year\"].max()}")
    del _m

📌 **[0.5]** ⚠️ cảnh báo cho file không tìm thấy không phải lỗi nghiêm trọng — ECDC chỉ dùng 
cho validation dashboard (có từ 2021). ERA5 cover 158/172 quốc gia (92%) do giới hạn KD-tree 
centroid matching. Dengue missing 88.9% rows là đúng — chỉ endemic countries mới có data.

# 🔍 SESSION 1 — LOAD & INSPECT RAW DATA
> **Mục tiêu:** Hiểu cấu trúc từng file nguồn, phát hiện vấn đề trước khi xử lý.
> **Input:** Raw CSV files từ RAW folder
> **Output:** Không có (chỉ exploration)
> **Có thể skip nếu:** Đã quen với cấu trúc data, muốn chạy thẳng SESSION 4+

In [ ]:
# [1.0] RESTART CELL — load tất cả raw files
flu      = pd.read_csv(FILES['flunet'], low_memory=False)
flu_meta = pd.read_csv(FILES['flu_meta'], low_memory=False)
dengue   = pd.read_csv(FILES['dengue'], low_memory=False)
ecdc_sen = pd.read_csv(FILES['ecdc_sen'], low_memory=False)
ecdc_ili = pd.read_csv(FILES['ecdc_ili'], low_memory=False)

for name, df in [('flu', flu), ('flu_meta', flu_meta), ('dengue', dengue),
                  ('ecdc_sen', ecdc_sen), ('ecdc_ili', ecdc_ili)]:
    print(f'{name}: shape={df.shape} | cols={list(df.columns[:5])}...')

📌 **[1.0]** FILES dict đã được define ở SESSION 0 — bao gồm đường dẫn đến tất cả raw files.
Load tập trung ở đây để các cell [1.1]–[1.4] chỉ cần tham chiếu biến, không load lại.
Nếu session restart, chạy lại [1.0] này trước khi chạy các cell inspect.

In [ ]:
# [1.1] Inspect FluNet
print('=== FluNet ===')
print(f'Shape: {flu.shape}')
print(f'Columns ({len(flu.columns)}):', list(flu.columns))
print(f'Year range: {flu["ISO_YEAR"].min()}-{flu["ISO_YEAR"].max()}')
print(f'Countries: {flu["COUNTRY_CODE"].nunique()}')
display(flu.head(3))

📌 **[1.1]** FluNet có 53 cột gồm nhiều subtype chi tiết (AH1N12009, AH3, BVIC...). Các cột
quan trọng sẽ dùng: INF_A, INF_B, COUNTRY_CODE, ISO_YEAR, ISO_WEEK.
Lưu ý: RSV và RSV_PROCESSED tồn tại nhưng khác đơn vị — sẽ xử lý ở SESSION 2.
PARAINFLUENZA present nhưng sẽ bị drop do missing rate cao (xem SESSION 2).

In [ ]:
# [1.2] Inspect OpenDengue
print('=== OpenDengue ===')
print(f'Shape: {dengue.shape}')
print('T_res distribution:')
print(dengue['T_res'].value_counts())
print(f'Year range (approx): {dengue["calendar_start_date"].dropna().iloc[0]} ... {dengue["calendar_start_date"].dropna().iloc[-1]}')
display(dengue.head(3))

📌 **[1.2]** T_res phân ra Week/Month/Year — chỉ dùng Week+Month để đủ granularity cho model tuần.
Date format của OpenDengue là MM/DD/YYYY (không nhất quán), cần `format='mixed'` khi parse.
Điều này đã được confirm và sẽ xử lý tường minh ở SESSION 3 và SESSION 5.

In [ ]:
# [1.3] Inspect ECDC Sentinel
print('=== ECDC Sentinel ===')
print(f'Shape: {ecdc_sen.shape}')
pathogen_col = [c for c in ecdc_sen.columns if 'pathogen' in c.lower() or 'Pathogen' in c]
if pathogen_col:
    print(f'Unique pathogens: {ecdc_sen[pathogen_col[0]].unique()}')
print(f'Countries: {ecdc_sen.iloc[:, 0].nunique()}')
display(ecdc_sen.head(3))

📌 **[1.3]** ECDC Sentinel có cả SARS-CoV-2 — cần filter khi dùng. Chỉ 30 quốc gia châu Âu,
chỉ có data từ 2021. Quyết định đã chốt: ECDC chỉ dùng cho validation và dashboard,
không dùng cho training (vì train period là 2010–2019).

In [ ]:
# [1.4] Inspect ECDC ILI
print('=== ECDC ILI ===')
print(f'Shape: {ecdc_ili.shape}')
age_col = [c for c in ecdc_ili.columns if 'age' in c.lower()]
ind_col = [c for c in ecdc_ili.columns if 'indicator' in c.lower()]
print(f'Age columns: {age_col}')
print(f'Indicator columns: {ind_col}')
yr_col = [c for c in ecdc_ili.columns if 'year' in c.lower()]
if yr_col:
    print(f'Year range: {ecdc_ili[yr_col[0]].min()}-{ecdc_ili[yr_col[0]].max()}')
display(ecdc_ili.head(3))

📌 **[1.4]** ECDC ILI có age groups đầy đủ (0–4, 5–14, 15–64, 65+, total) — hữu ích cho
dashboard chi tiết khi hiển thị breakdown theo nhóm tuổi.
Cũng chỉ có từ 2021 nên không dùng cho training.

# 🔎 SESSION 2 — DATA QUALITY CHECK
> **Mục tiêu:** Missing rate, coverage quốc gia/năm, phát hiện anomaly.
> **Input:** raw DataFrames từ SESSION 1 (hoặc load lại từ [2.0])
> **Output:** Không có — kết quả dẫn đến các quyết định tiền xử lý

In [ ]:
# [2.0] RESTART CELL — load flu và dengue từ disk
flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)
print(f'flu: {flu.shape}')
print(f'dengue: {dengue.shape}')

📌 **[2.0]** Chỉ load 2 file cần cho session này để tiết kiệm RAM.
Nếu đã chạy SESSION 1 và biến còn tồn tại, cell này không bị lỗi — chỉ reload mới nhất từ disk.

In [ ]:
# [2.1] FluNet — Missing rate cho các cột quan trọng
check_cols = ['INF_A', 'INF_B', 'INF_ALL', 'RSV', 'RSV_PROCESSED',
              'PARAINFLUENZA', 'ILI_ACTIVITY', 'SPEC_PROCESSED_NB']
# Only keep cols that exist in dataframe
check_cols = [c for c in check_cols if c in flu.columns]
missing_pct = flu[check_cols].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(8, 5))
missing_pct.plot(kind='barh', ax=ax, color='salmon', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Missing (%)')
ax.set_title('FluNet — Missing Rate per Column')
for bar, val in zip(ax.patches, missing_pct):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

📌 **[2.1]** Biểu đồ xác nhận missing rate khớp với các quyết định đã chốt:

| Cột | Missing | Quyết định |
|---|---|---|
| PARAINFLUENZA | 85.5% | ❌ Loại |
| RSV_PROCESSED | 81.9% | ❌ Loại (trùng RSV) |
| ILI_ACTIVITY | 60.0% | ⚠️ Categorical |
| RSV | 52.1% | ⚠️ Đã có `rsv_cases` |
| INF_ALL | 44.3% | ❌ KHÔNG dùng target |
| INF_A | 12.9% | ✅ Dùng |
| INF_B | 12.1% | ✅ Dùng |
| SPEC_PROCESSED_NB | 7.0% | ✅ Tổng mẫu xét nghiệm |

**Target:** `inf_total = INF_A + INF_B` với `fillna(0)` (missing = không báo cáo).

**Bài học:** Cột > 50% missing → cờ đỏ, cần lý do mạnh để giữ.

In [ ]:
# [2.2] FluNet — Coverage theo năm (số quốc gia báo cáo)
coverage = flu.groupby('ISO_YEAR')['COUNTRY_CODE'].nunique().reset_index()
coverage.columns = ['year', 'n_countries']

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(coverage['year'], coverage['n_countries'], color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(TRAIN_START - 0.5, color='green', lw=2, ls='--', label=f'Train start ({TRAIN_START})')
ax.axvline(TRAIN_END + 0.5, color='red', lw=2, ls='--', label=f'Train end ({TRAIN_END})')
ax.set_xlabel('Year')
ax.set_ylabel('Countries reporting')
ax.set_title('FluNet — Country Coverage by Year')
ax.legend()
plt.tight_layout()
plt.show()

📌 **[2.2]** Coverage ổn định từ 2010 trở đi (123 → 167 nước). Trước 2010 chỉ ~60-100 nước → dùng từ 2010 là an toàn.

**⚠️ Đính chính so với note cũ:** Giai đoạn 2020–2021 coverage **KHÔNG GIẢM** (vẫn 166–167). Lý do exclude thực sự:
1. **NPI effects** (giãn cách, khẩu trang) → giảm transmission flu artificially
2. **Reporting bias** — ca hô hấp ưu tiên test COVID, không test flu
3. **Pattern bị méo** — flu 2020 giảm ~99% toàn cầu, không do thời tiết

Năm 2009 spike (102 nước, +44 so 2008) do **H1N1 pandemic** ép WHO scaling reporting.

In [ ]:
# [2.3] OpenDengue — Missing & coverage
print('dengue_total missing rate:', round(dengue['dengue_total'].isnull().mean()*100, 1), '%')
print('Year range:', dengue['calendar_start_date'].dropna().iloc[0], '...',
      dengue['calendar_start_date'].dropna().iloc[-1])

fig, ax = plt.subplots(figsize=(6, 6))
tres_counts = dengue['T_res'].value_counts()
ax.pie(tres_counts, labels=tres_counts.index, autopct='%1.1f%%', startangle=90,
       colors=['#2ecc71','#3498db','#e74c3c'])
ax.set_title('OpenDengue — T_res Distribution')
plt.tight_layout()
plt.show()

📌 **[2.3]** OpenDengue raw — quan sát:

- **T_res pie chart**: 77.8% Week, 11.7% Year, 10.5% Month → confirm chọn `T_res ∈ {Week, Month}` (88.3% data)
- **dengue_total missing 0%** trên rows đã filter
- **⚠️ Year range output bất thường**: `9/5/2021 ... 7/21/2024` — chỉ thấy 2021-2024 thay vì 1990-2024

**Khả năng:** `calendar_start_date` kiểu **string** chứ không datetime → `.min()/.max()` sort lexicographic ("9/5/2021" > "10/1/2020" theo bảng chữ cái). Sẽ verify ở SESSION 3 mở rộng bước 1 (schema check).

OpenDengue có data từ 1990s — output [3.4] sau đó xác nhận Brazil 7.15M ca trong 2010-2019.

In [ ]:
# [2.4] ERA5 — Coverage check
era5 = pd.read_csv(ERA5_FILE)
print(f'ERA5 shape: {era5.shape}')
print(f'Countries: {era5["iso3"].nunique()}')

missing_era5 = era5.drop(columns=['iso3','iso_year','iso_week']).isnull().mean() * 100
missing_era5 = missing_era5[missing_era5 > 0].sort_values(ascending=False)

if len(missing_era5) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_era5.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing (%)')
    ax.set_title('ERA5 — Missing Rate per Variable')
    plt.tight_layout()
    plt.show()
else:
    print('ERA5: khong co missing values')

📌 **[2.4]** ERA5 cover đầy đủ **197 quốc gia**, 102,440 rows = 197 × 520 tuần (2010–2019), 21 columns. Không có missing values — pixel mask averaging (trung bình tất cả grid points trong biên giới quốc gia) cover tốt hơn KD-tree nearest centroid. Sau khi merge với FluNet (172 quốc gia), 154/172 quốc gia có weather data — 18 quốc gia FluNet là lãnh thổ đặc biệt không có trong Natural Earth shapefile.

# 📊 SESSION 3 — EDA: SEASONALITY & TRENDS
> **Mục tiêu:** Xác nhận pattern mùa vụ rõ ràng — điều kiện cần để train model.
> **Input:** Raw FluNet + OpenDengue
> **Output:** Không có file — chỉ visualizations

In [ ]:
# [3.0] RESTART CELL + setup train range
flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)

flu_train = flu[flu['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
flu_train['inf_total'] = flu_train['INF_A'].fillna(0) + flu_train['INF_B'].fillna(0)
print(f'flu_train: {flu_train.shape} | years: {TRAIN_START}-{TRAIN_END}')

📌 **[3.0]** TRAIN_START=2010, TRAIN_END=2019 đã chốt ở SESSION 0.
flu_train ở đây dùng riêng cho EDA, không phải feature matrix cuối cùng.
inf_total = INF_A + INF_B là target chính — INF_ALL bị bỏ do missing 44%.

In [ ]:
# [3.1] FluNet — Global trend + seasonality
flu_weekly = flu_train.groupby(['ISO_YEAR','ISO_WEEK'])['inf_total'].sum().reset_index()
flu_weekly['time_idx'] = flu_weekly['ISO_YEAR'] + flu_weekly['ISO_WEEK'] / 53
flu_season = flu_train.groupby('ISO_WEEK')['inf_total'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

axes[0].plot(flu_weekly['time_idx'], flu_weekly['inf_total'], lw=1.2, color='steelblue')
axes[0].set_title('Global Influenza Cases per Week (2010-2019)')
axes[0].set_xlabel('Year')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

axes[1].bar(flu_season['ISO_WEEK'], flu_season['inf_total'], color='steelblue', alpha=0.8)
axes[1].set_title('Average Seasonality by ISO Week (2010-2019)')
axes[1].set_xlabel('ISO Week')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.show()

📌 **[3.1]** Trend phẳng (không tăng mạnh) là tốt — cho thấy data ổn định, không bị confound bởi
reporting bias tăng dần theo năm. Seasonality rõ peak tuần 1–10 (mùa đông bắc bán cầu).
Đây là pattern chính model cần học; weather features cung cấp signal để predict timing và amplitude.

In [ ]:
# [3.2] FluNet — 5 quốc gia đại diện
countries = ['VNM', 'USA', 'GBR', 'BRA', 'AUS']
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)

for ax, iso in zip(axes, countries):
    df_c = flu_train[flu_train['COUNTRY_CODE'] == iso]
    season_c = df_c.groupby('ISO_WEEK')['inf_total'].mean()
    ax.bar(season_c.index, season_c.values, color='steelblue', alpha=0.8)
    ax.set_title(iso, fontsize=12)
    ax.set_xlabel('ISO Week')
    if ax is axes[0]:
        ax.set_ylabel('Avg cases')

plt.suptitle('Influenza Seasonality by Country (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.2]** Pattern khác nhau rõ theo khí hậu: BRA và AUS peak tháng 6–8 (nam bán cầu),
USA/GBR peak tháng 12–2. VNM có 2 peak nhỏ hơn (nhiệt đới).
Model train per-country tự học được sự khác biệt này qua iso3 encoding và lag features.

In [ ]:
# [3.3] Dengue — Filter + parse date
dengue_wm = dengue[dengue['T_res'].isin(['Week','Month'])].copy()
dengue_wm['date_parsed'] = pd.to_datetime(dengue_wm['calendar_start_date'], format='mixed', dayfirst=False)
iso_cal = dengue_wm['date_parsed'].dt.isocalendar()
dengue_wm['ISO_YEAR'] = iso_cal.year.astype(int)
dengue_wm['ISO_WEEK'] = iso_cal.week.astype(int)
dengue_train = dengue_wm[dengue_wm['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
dengue_train['dengue_log'] = np.log1p(dengue_train['dengue_total'])
print(f'dengue_train: {dengue_train.shape}')
print(f'Countries: {dengue_train["ISO_A0"].nunique()}')

📌 **[3.3]** `format='mixed'` cần thiết vì OpenDengue date không nhất quán (MM/DD/YYYY).
log1p ngay ở đây để visualization dùng log scale — dengue_total bị dominated bởi Brazil
với ~7.15M ca, chiếm 51.4% tổng global (Brazil: 7,152,784 / Global: 13,907,091 ca). log1p giúp thấy pattern của các nước khác.

In [ ]:
# [3.4] Dengue — Trend + seasonality (raw vs log)
by_year_raw = dengue_train.groupby('ISO_YEAR')['dengue_total'].sum()
by_week_raw = dengue_train.groupby('ISO_WEEK')['dengue_total'].mean()
by_year_log = dengue_train.groupby('ISO_YEAR')['dengue_log'].sum()
by_week_log = dengue_train.groupby('ISO_WEEK')['dengue_log'].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].bar(by_year_raw.index, by_year_raw.values, color='coral')
axes[0,0].set_title('Raw — by Year')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

axes[0,1].bar(by_week_raw.index, by_week_raw.values, color='coral')
axes[0,1].set_title('Raw — by ISO Week')

axes[1,0].bar(by_year_log.index, by_year_log.values, color='#27ae60')
axes[1,0].set_title('Log1p — by Year')

axes[1,1].bar(by_week_log.index, by_week_log.values, color='#27ae60')
axes[1,1].set_title('Log1p — by ISO Week')

plt.suptitle('Dengue Trend & Seasonality (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.4]** Raw bị dominated bởi Brazil (7.15M ca, 51.4% tổng global). Log scale cho thấy pattern của các nước
khác rõ hơn — đây là bằng chứng thực nghiệm cho quyết định dùng log1p làm target.
By-week log plot cho thấy seasonality thực sự khi không bị Brazil che khuất.

In [ ]:
# [3.5] Dengue — Top 5 (loại Brazil)
dengue_no_bra = dengue_train[dengue_train['ISO_A0'] != 'BRA']
top5 = dengue_no_bra.groupby('ISO_A0')['dengue_total'].sum().nlargest(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
for ax, iso in zip(axes, top5):
    df_c = dengue_no_bra[dengue_no_bra['ISO_A0'] == iso]
    season_c = df_c.groupby('ISO_WEEK')['dengue_total'].mean()
    ax.bar(season_c.index, season_c.values, color='#27ae60', alpha=0.8)
    ax.set_title(iso, fontsize=12)
    ax.set_xlabel('ISO Week')

plt.suptitle('Dengue Seasonality — Top 5 ex-Brazil (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.5]** Peak mùa mưa khác nhau theo khu vực: Đông Nam Á (Philippines, Thái Lan, Indonesia)
peak tuần 25–45; Trung Mỹ (Mexico, Colombia) peak tuần 1–20.
Sự đa dạng này xác nhận cần per-country model với weather lag features — không dùng global model đơn.

📌 **[3.6]** Heatmap (Year × ISO Week) cho Việt Nam — pattern khác hẳn ôn đới:

- **Không có winter peak** (T12-T2) như USA/GBR — đặc trưng tropical climate
- **Đa đỉnh**: mid-year (W22-W34, tức T6-T8) và late-year (W39-W44, tức T9-T11)
- **2010-2011**: peak mạnh (max 65-70 cases) — hậu pandemic H1N1, surveillance aggressive
- **2012-2019**: max giảm về 30-50 cases/tuần — under-reporting hoặc dịch nhẹ
- **Year-to-year variability cao** → khó forecast bằng seasonal average

**Hàm ý cho ML approach v3:**
- Per-country Endemic Channel baseline là **bắt buộc** — VN không thể dùng cùng threshold với Brazil (10,000+ ca/tuần)
- Walk-forward CV cần monitor class balance per country (VN có thể fold toàn Low)
- Tránh train global pooled không phân country → Brazil dominate, mất signal VN

# 🌦️ SESSION 4 — ERA5 DOWNLOAD & PROCESS
> **NẶNG — Chỉ chạy nếu ERA5_FILE chưa tồn tại**
> **Input:** ERA5 NetCDF files + Natural Earth shapefile
> **Output:** era5_weekly_2010_2019_final.csv
> ✅ File đã tồn tại — SESSION NÀY ĐÃ HOÀN THÀNH

In [ ]:
# [4.0] Idempotent guard
if ERA5_FILE.exists():
    era5 = pd.read_csv(ERA5_FILE)
    print(f'ERA5 da co: {ERA5_FILE.name}')
    print(f'Shape: {era5.shape} | Countries: {era5["iso3"].nunique()} | Years: {era5["iso_year"].min()}-{era5["iso_year"].max()}')
    print('SESSION 4 hoan thanh - skip xuong SESSION 5')
else:
    print('ERA5 chua co - can chay tu [4.1]')

📌 **[4.0]** ERA5 đã được xử lý và lưu vào era5_weekly_2010_2019_final.csv với 17 biến khí hậu.
Script gốc dùng KD-tree nearest centroid mapping từ lưới ERA5 (0.25°×0.25°) sang centroid
quốc gia theo Natural Earth 50m. Chỉ cần chạy lại nếu xóa file hoặc cần thêm biến mới.

In [ ]:
# [4.1] (Conditional) Process ERA5 nếu cần
# Placeholder — chỉ chạy nếu ERA5_FILE chua ton tai
# Script day du nam o: scripts/process_era5.py
# Thoi gian uoc tinh: 2-3 gio cho 10 nam du lieu

ERA5_VARS_DOWNLOAD = [
    '2m_temperature', 'total_precipitation', '2m_dewpoint_temperature',
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'surface_solar_radiation_downwards', 'surface_thermal_radiation_downwards',
    'total_cloud_cover', 'mean_sea_level_pressure', 'boundary_layer_height',
    'total_evaporation', 'convective_precipitation', 'large_scale_precipitation',
    'soil_temperature_level_1', 'volumetric_soil_water_layer_1',
    'surface_net_solar_radiation', 'total_column_water_vapour'
]
print(f'{len(ERA5_VARS_DOWNLOAD)} variables to download')
print('Xem scripts/process_era5.py de chay day du')

📌 **[4.1]** Download ERA5 qua CDS API mất ~2–3 giờ cho 10 năm × 17 biến.
Đã được lưu checkpoint theo từng năm (era5_weekly_era5_YYYY_checkpoint.csv) để tránh mất công
nếu bị ngắt giữa chừng. Cuối cùng concat tất cả vào final CSV.

# 🔗 SESSION 5 — PREPROCESSING & MERGE
> **Mục tiêu:** Chuẩn hóa 3 nguồn về key iso3+iso_year+iso_week, merge → master_weekly.csv
> **Input:** FluNet + ERA5 + OpenDengue + Malaria (GHO)
> **Output:** dataset/processed/master_weekly_2010_2019.csv
> ✅ File đã tồn tại — SESSION NÀY ĐÃ HOÀN THÀNH

In [ ]:
# [5.0] Idempotent guard + verify
if MASTER_FILE.exists():
    master = pd.read_csv(MASTER_FILE)
    print(f'master_weekly da co: {MASTER_FILE.name}')
    print(f'Shape: {master.shape}')
    print(f'Countries: {master["iso3"].nunique()} | Years: {master["iso_year"].min()}-{master["iso_year"].max()}')
    print(f'Columns: {list(master.columns)}')
    print('SESSION 5 hoan thanh - chuyen sang SESSION 6')
else:
    print('master_weekly chua co - can chay tu [5.1]')

📌 **[5.0]** master_weekly_2010_2019.csv là kết quả merge của FluNet (anchor) LEFT JOIN ERA5
LEFT JOIN OpenDengue LEFT JOIN Malaria. 64,949 rows × 27 columns, 172 quốc gia.
Đây là file đầu vào cho toàn bộ pipeline ML từ SESSION 6 trở đi.

In [ ]:
# [5.1] (Conditional) Merge logic — nếu cần tái tạo
# Chay cell nay ONLY neu MASTER_FILE chua ton tai

flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)
era5   = pd.read_csv(ERA5_FILE)

# FluNet preprocessing
flu_proc = flu[flu['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
flu_proc['inf_cases'] = flu_proc['INF_A'].fillna(0) + flu_proc['INF_B'].fillna(0)
flu_proc['rsv_cases'] = flu_proc['RSV'].fillna(0)
flu_proc = flu_proc.rename(columns={'COUNTRY_CODE':'iso3','ISO_YEAR':'iso_year','ISO_WEEK':'iso_week'})
flu_proc = flu_proc[['iso3','iso_year','iso_week','inf_cases','rsv_cases']]

# ERA5 preprocessing
era5_proc = era5[era5['iso_year'].between(TRAIN_START, TRAIN_END)].copy()

# Dengue preprocessing
dengue_wm = dengue[dengue['T_res'].isin(['Week','Month'])].copy()
dengue_wm['date_parsed'] = pd.to_datetime(dengue_wm['calendar_start_date'], format='mixed', dayfirst=False)
iso_cal = dengue_wm['date_parsed'].dt.isocalendar()
dengue_wm['iso_year'] = iso_cal.year.astype(int)
dengue_wm['iso_week'] = iso_cal.week.astype(int)
dengue_proc = dengue_wm[dengue_wm['iso_year'].between(TRAIN_START, TRAIN_END)].copy()
dengue_proc['dengue_total'] = dengue_proc['dengue_total'].fillna(0)
dengue_proc['dengue_log1p'] = np.log1p(dengue_proc['dengue_total'])
dengue_proc = dengue_proc.rename(columns={'ISO_A0':'iso3'})
dengue_proc = dengue_proc.groupby(['iso3','iso_year','iso_week'], as_index=False).agg(
    dengue_total=('dengue_total','sum'), dengue_log1p=('dengue_log1p','mean')
)

# Merge: FluNet (anchor) LEFT JOIN ERA5 LEFT JOIN Dengue
master = flu_proc.merge(era5_proc, on=['iso3','iso_year','iso_week'], how='left')
master = master.merge(dengue_proc, on=['iso3','iso_year','iso_week'], how='left')
master['dengue_total'] = master['dengue_total'].fillna(0)
master['dengue_log1p'] = master['dengue_log1p'].fillna(0)

master.to_csv(MASTER_FILE, index=False)
print(f'Saved {len(master):,} rows -> {MASTER_FILE.name}')

📌 **[5.1]** FluNet là anchor (LEFT JOIN) — giữ nguyên tất cả 172 quốc gia × 10 năm × 52 tuần.
fillna(0) sau merge cho dengue_total vì quốc gia không có dengue report = 0 ca thực, không phải missing.
ERA5 join theo iso3+iso_year+iso_week — 14 quốc gia không có ERA5 sẽ có NaN cho weather features.

In [ ]:
# [5.2] Missing rate — kiểm tra NaN từng cột
master = pd.read_csv(MASTER_FILE)

miss = master.isnull().mean().sort_values(ascending=False) * 100
print('=== NaN rate (%) ===')
print(miss[miss > 0].to_string())

has_weather = (master.groupby('iso3')['temp_c'].count() > 0).sum()
total_countries = master['iso3'].nunique()
print(f'\nNuoc co weather: {has_weather}/{total_countries}')
print(f'NaN weather tong the: {master["temp_c"].isnull().mean()*100:.1f}%')

📌 **[5.2]** Kiem tra ty le NaN tung cot trong master_weekly.

**Nguong chap nhan:**
- `temp_c` va cac weather cols: NaN < 20% tong rows (ERA5 chi co 154/172 nuoc)
- `inf_cases`: NaN = 0 (FluNet la anchor — LEFT JOIN nen moi row deu co)
- `dengue_log1p`: NaN cao la binh thuong — chi endemic countries moi co data

Neu weather NaN > 30% → kiem tra lai qua trinh merge ERA5 o SESSION 4.
Neu inf_cases co NaN → loi nghiem trong, phai debug [5.1] merge logic.

In [ ]:
# [5.3] Phan phoi gia tri — phat hien outlier weather
weather_cols = ['temp_c', 'humidity_pct', 'precip_mm', 'solar_wm2', 'dewpoint_c']
print('=== Describe weather vars ===')
print(master[weather_cols].describe().round(2).to_string())

# Flag outlier ro rang
flags = {
    'temp_c':       (master['temp_c'] < -60) | (master['temp_c'] > 60),
    'humidity_pct': (master['humidity_pct'] < 0) | (master['humidity_pct'] > 105),
    'precip_mm':    master['precip_mm'] < 0,
    'solar_wm2':    master['solar_wm2'] < 0,
}
print('\n=== Outlier rows ===')
for col, mask in flags.items():
    n = mask.sum()
    if n > 0:
        print(f'  {col}: {n} rows ngoai nguong')
    else:
        print(f'  {col}: OK')

📌 **[5.3]** Kiem tra phan phoi gia tri weather de phat hien loi don vi hoac processing bug.

**Nguong hop ly:**
- `temp_c`: -60 den +60 C (gia tri ngoai pham vi nay la loi ERA5 extraction)
- `humidity_pct`: 0-105% (ERA5 co the ra 100.x% do interpolation, chap nhan duoc)
- `precip_mm`: >= 0 (am la loi tuyet doi)
- `solar_wm2`: >= 0 (am la loi don vi — phai la W/m2)

Neu co outlier → kiem tra lai `process_era5.py` phan unit conversion (ERA5 tra
precipitation theo m/s, phai nhan 1000 * 3600 de ra mm/h roi x 24*7 ra mm/week).

In [ ]:
# [5.4] Coverage theo nam — so nuoc bao cao cum moi nam
import matplotlib.pyplot as plt

flu_cov = (master[master['inf_cases'] > 0]
           .groupby('iso_year')['iso3'].nunique()
           .reindex(range(2010, 2020), fill_value=0))

weather_cov = (master.dropna(subset=['temp_c'])
               .groupby('iso_year')['iso3'].nunique()
               .reindex(range(2010, 2020), fill_value=0))

print('=== So nuoc co data tung nam (2010-2019) ===')
print('Nam   | Flu report | Co weather')
print('------+------------+-----------')
for yr in range(2010, 2020):
    print(f'{yr}  | {flu_cov[yr]:>10} | {weather_cov[yr]:>10}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(flu_cov.index - 0.2, flu_cov.values, 0.4, label='Flu reporting', color='steelblue')
ax.bar(weather_cov.index + 0.2, weather_cov.values, 0.4, label='Has weather', color='coral')
ax.axhline(120, ls='--', color='gray', lw=0.8, label='Threshold 120 nuoc')
ax.set_xlabel('Nam'); ax.set_ylabel('So nuoc'); ax.set_title('Coverage: Flu report vs Weather data')
ax.legend(); plt.tight_layout(); plt.show()

📌 **[5.4]** Bar chart so nuoc co flu reporting va weather theo tung nam (2010-2019).

**Ky vong:**
- Flu report: > 120 nuoc moi nam (WHO FluNet coverage on dinh 2010-2019)
- Has weather: on dinh quanh 154 nuoc (ERA5 KD-tree coverage cua du an)

Neu mot nam nao do dot ngot thap hon (< 100 nuoc flu, < 130 weather) → kiem tra
lai SESSION 4 ERA5 processing hoac FluNet filter logic o [1.x].
Nam 2009 va 2020-2021 bi loai tu dau (COVID + H1N1 disruption) nen khong xuat hien.

In [ ]:
# [5.5] Sanity check target variables

# Influenza: ty le zero rows
zero_flu = (master['inf_cases'] == 0).mean() * 100
print(f'Flu zero rows: {zero_flu:.1f}%  (ky vong ~70-75%)')

# Dengue: so nuoc endemic
dengue_countries = master[master['dengue_log1p'] > 0]['iso3'].nunique()
print(f'Dengue endemic countries: {dengue_countries}  (ky vong 15-25)')

# Influenza max — check bat thuong
flu_max = master.nlargest(5, 'inf_cases')[['iso3','iso_year','iso_week','inf_cases']]
print('\n=== Top 5 flu weeks ===')
print(flu_max.to_string(index=False))

# Dengue max
den_max = master.nlargest(5, 'dengue_log1p')[['iso3','iso_year','iso_week','dengue_log1p','dengue_total']]
print('\n=== Top 5 dengue weeks ===')
print(den_max.to_string(index=False))

📌 **[5.5]** Kiem tra gia tri dau ra 2 bien muc tieu.

- **Flu zero rows ~70-75%**: binh thuong vi cum co seasonal pattern manh — ngoai
  mua dong nhieu nuoc bao cao 0. Neu < 60% → co the FluNet filter bi loi.
  Neu > 80% → kiem tra lai fillna(0) cho INF_A/INF_B co lam sai lenh (bao nhieu
  nuoc thuc su co data vs nuoc bao cao 0).
- **Dengue endemic countries 15-25**: phu hop voi vung nhiet doi/can nhiet doi co
  muoi Aedes. Neu > 40 → co the filter `dengue_log1p > 0` dang loi.
- **Top 5**: kiem tra xem co gia tri bat hop ly khong (flu > 500,000 ca/tuan o mot
  quoc gia nho → suspect). Brazil la top dengue thuong xuyen — binh thuong.

In [ ]:
# [5.6] Kiem tra ERA5 monthly means — seasonal sanity

# Verify temp_range_c = 0 (limitation cua monthly means)
if 'temp_range_c' in master.columns:
    zero_range = (master['temp_range_c'] == 0).mean() * 100
    print(f'temp_range_c = 0: {zero_range:.1f}%  (ky vong ~100% vi ERA5 monthly means)')
else:
    print('temp_range_c: khong co trong master (OK neu dung 25-col version)')

# Seasonal check: USA thang 1 vs thang 7
usa = master[master['iso3'] == 'USA'].copy()
if len(usa) > 0:
    jan = usa[usa['iso_week'].between(1, 4)]['temp_c'].mean()
    jul = usa[usa['iso_week'].between(27, 30)]['temp_c'].mean()
    print(f'USA Jan avg: {jan:.1f}C  (ky vong -5 den 5C)')
    print(f'USA Jul avg: {jul:.1f}C  (ky vong 20-25C)')
    ok_usa = jan < 5 and jul > 15
    print(f'USA seasonal check: {"OK" if ok_usa else "FAIL — kiem tra ERA5 extraction"}')
else:
    print('USA: khong co trong master — can kiem tra SESSION 4')

# Seasonal check: VNM (Vietnam — nhiet doi, it seasonal variation)
vnm = master[master['iso3'] == 'VNM'].copy()
if len(vnm) > 0:
    jan_v = vnm[vnm['iso_week'].between(1, 4)]['temp_c'].mean()
    jul_v = vnm[vnm['iso_week'].between(27, 30)]['temp_c'].mean()
    print(f'VNM Jan avg: {jan_v:.1f}C  (ky vong 15-22C)')
    print(f'VNM Jul avg: {jul_v:.1f}C  (ky vong 27-32C)')
else:
    print('VNM: khong co trong master')

print('\n=== Checklist SESSION 5 ===')
weather_nan = master['temp_c'].isnull().mean() * 100
n_countries = master['iso3'].nunique()
zero_flu_ok = 65 < (master['inf_cases'] == 0).mean() * 100 < 80
print(f'  weather NaN < 20%: {"OK" if weather_nan < 20 else "FAIL"} ({weather_nan:.1f}%)')
print(f'  coverage > 130 nuoc: {"OK" if n_countries > 130 else "FAIL"} ({n_countries})')
print(f'  flu zero rows 65-80%: {"OK" if zero_flu_ok else "CHECK"} ({(master["inf_cases"]==0).mean()*100:.1f}%)')
print(f'  USA seasonal check: {"OK" if ok_usa else "FAIL"}' if len(usa) > 0 else '  USA seasonal: SKIP (no USA data)')

📌 **[5.6]** Kiem tra cuoi cung xac nhan ERA5 monthly means dang hoat dong dung.

- **temp_range_c = 100%**: la han che co ban cua ERA5 monthly means — moi tuan trong
  cung thang se co gia tri temp giong nhau (monthly average duoc ffill sang 4-5 tuan).
  Day la limitation da biet, can neu ro trong bao cao.
- **USA seasonal check**: xac nhan ERA5 extraction capture duoc seasonal signal that.
  Jan < 5C va Jul > 15C la nguong toi thieu — neu fail tuc ERA5 unit conversion sai.
- **VNM check**: nhiet doi nen temp on dinh hon (khong co mua dong ro net), nhung
  van phai co chenh lech ~10C giua mua kho va mua mua.

Neu tat ca checklist OK -> proceed to SESSION 6 (CCF analysis).
Neu bat ky check nao FAIL -> quay lai SESSION 4 kiem tra ERA5 processing.

---

## 🔬 SESSION 6 — EDA MỞ RỘNG (8 BƯỚC)

> Tiếp nối SESSION 3 cơ bản. Trước khi vào feature engineering, phải pass 8 bước:
> 1. **Schema check** đầy đủ master_weekly (dtype, nunique, sample rows)
> 2. **Missing analysis** chi tiết — pattern theo (country × year)
> 3. **Distribution** — histogram log-scale per disease, boxplot per country
> 4. **Outlier** — IQR + z-score + domain threshold
> 5. **Time coverage** — gap detection per (iso3, year)
> 6. **CV split logic** — verify walk-forward không leakage
> 7. **Sanity check dịch tễ** — top countries, peak mùa vs lý thuyết
> 8. **Endemic Channel preview** — preview label trước khi gen full ở SESSION 8


In [ ]:
# [6.0] BƯỚC 1 — Schema check chi tiết master_weekly
# Mục tiêu: hiểu rõ TYPE, NUNIQUE, SAMPLE của từng cột trước khi vào EDA sâu

master = pd.read_csv(MASTER_FILE)

print('=' * 70)
print(f'Shape: {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'Memory: {master.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print('=' * 70)

# Build schema table
schema = pd.DataFrame({
    'dtype': master.dtypes.astype(str),
    'non_null': master.notna().sum(),
    'null_%': (master.isna().mean() * 100).round(1),
    'nunique': master.nunique(),
    'min': [master[c].min() if pd.api.types.is_numeric_dtype(master[c]) else '-' for c in master.columns],
    'max': [master[c].max() if pd.api.types.is_numeric_dtype(master[c]) else '-' for c in master.columns],
    'sample': [master[c].dropna().iloc[0] if master[c].notna().any() else None for c in master.columns]
})
print('\n--- SCHEMA ---')
print(schema.to_string())

# Categorize columns by role
keys = [c for c in master.columns if c in ['iso3','iso_year','iso_week']]
targets = [c for c in master.columns if 'cases' in c or 'total' in c or 'log1p' in c]
weather = [c for c in master.columns if c not in keys + targets]

print(f'\n--- COLUMN ROLES ---')
print(f'Keys     ({len(keys):2d}): {keys}')
print(f'Targets  ({len(targets):2d}): {targets}')
print(f'Weather  ({len(weather):2d}): {weather}')

# Sample 5 rows from different countries
print('\n--- SAMPLE 5 ROWS (different countries) ---')
sample_countries = ['VNM','USA','BRA','IND','GBR']
sample_rows = master[master['iso3'].isin(sample_countries)].groupby('iso3').head(1)
print(sample_rows.to_string())


*(note sau khi chay)*


In [ ]:
# [6.1] BƯỚC 2 — Missing pattern: tìm quốc gia nào không có ERA5

weather_cols = [c for c in master.columns if c not in
                ['iso3','iso_year','iso_week','inf_cases','rsv_cases','dengue_total','dengue_log1p']]

missing_mask = master[weather_cols[0]].isna()
missing_rows = master[missing_mask]

print(f'Tổng rows missing weather: {missing_mask.sum()} ({missing_mask.mean()*100:.1f}%)')

# Quốc gia nào bị missing?
missing_by_country = missing_rows.groupby('iso3').size().reset_index(name='missing_weeks')
missing_by_country['total_weeks'] = master.groupby('iso3').size().reindex(missing_by_country['iso3']).values
missing_by_country['missing_%'] = (missing_by_country['missing_weeks'] / missing_by_country['total_weeks'] * 100).round(1)
missing_by_country = missing_by_country.sort_values('missing_%', ascending=False)

print(f'\nSố quốc gia có missing weather: {len(missing_by_country)}')
print('\n--- Danh sách quốc gia missing weather ---')
print(missing_by_country.to_string(index=False))

full_missing  = missing_by_country[missing_by_country['missing_%'] == 100.0]
partial_missing = missing_by_country[missing_by_country['missing_%'] < 100.0]
print(f'\nMissing 100% (không có ERA5): {len(full_missing)} nước → {list(full_missing["iso3"])}')
print(f'Missing một phần ({len(partial_missing)} nước):')
if len(partial_missing) > 0:
    print(partial_missing.to_string(index=False))


📌 **[6.1]** Phân tích missing pattern của ERA5 weather sau khi merge vào master_weekly.

**Kết quả:** 6,645 rows missing (8.5%) — nghe nhiều nhưng là **systematic missing**, không phải random, nên cách xử lý rõ ràng.

---

**Nhóm 1 — 18 nước missing 100% (~6,496 rows):**

| Loại | iso3 | Nguyên nhân |
|---|---|---|
| Đảo siêu nhỏ | BLM, BRB, BMU, DMA, GLP, GUF, KNA, MAF, MDV, MTQ, TCA | Diện tích < 1,000 km² → không có ERA5 grid point nào rơi vào lãnh thổ (ERA5 resolution 0.25° ≈ 28 km) |
| UK pseudo-codes | X09, X10, X11, X12 | WHO tạo mã riêng cho England/Wales/Scotland/Bắc Ireland — không phải ISO 3166 chuẩn, không có centroid lat/lon |
| Kosovo | XKX | UN non-member state, nhiều geospatial database không có centroid |
| Malta | MLT | 316 km², đảo nhỏ — tương tự nhóm đảo |
| Singapore | SGP | Đảo ~728 km², centroid có thể rơi vào vùng biển trong ERA5 grid |

→ **Quyết định: drop 18 nước này** khi train. Không thể impute weather cho nước chưa từng có ERA5 record. Sau drop còn **154 nước** (172 - 18).

---

**Nhóm 2 — 105 nước missing 0.2–0.6% (149 rows tổng cộng):**

- Hầu hết missing **exactly 1–2 tuần** — rõ ràng không phải random
- Nhiều nước Đông Âu (MKD, ALB, HUN, CZE...) missing 2 tuần; nước lớn (USA, BRA, CHN) missing 1 tuần
- **Nguyên nhân:** ISO week 53 edge case — một số năm có 53 tuần (e.g., 2015, 2020), ERA5 processing theo calendar year có thể tạo gap ở boundary. Đây là **processing artifact**, không phải real data loss.

→ **Quyết định: forward-fill** (ffill) từ tuần liền trước. 149 / 78,213 = 0.2% ảnh hưởng — không đáng kể.

---

**Tóm tắt hành động:**

| Nhóm | Rows | Hành động |
|---|---|---|
| 18 nước, 100% missing | 6,496 | Drop khỏi training |
| 105 nước, 1–3 tuần | 149 | Forward-fill từ tuần trước |

Sau khi xử lý: **~71,717 rows** sạch, sẵn sàng cho feature engineering.


In [ ]:
# [6.2] BƯỚC 3 — Distribution: histogram + boxplot per disease
# Mục tiêu: hiểu phân phối thực tế của target, phát hiện zero-inflation và long-tail

master = pd.read_csv(MASTER_FILE)

# ── Bỏ 18 nước 100% missing để distribution thực tế hơn ──
NO_ERA5_COUNTRIES = [
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
]
master_clean = master[~master['iso3'].isin(NO_ERA5_COUNTRIES)].copy()
print(f'Sau drop 18 nước: {master_clean.shape[0]:,} rows, {master_clean["iso3"].nunique()} countries')

# ── FLU ──────────────────────────────────────────────────────────────────────
flu_cases = master_clean['inf_cases'].dropna()
flu_nonzero = flu_cases[flu_cases > 0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# [A] Histogram raw (tất cả rows)
axes[0,0].hist(flu_cases, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
axes[0,0].set_title('Flu: raw (incl. zeros)')
axes[0,0].set_xlabel('inf_cases'); axes[0,0].set_ylabel('count')
zero_pct = (flu_cases == 0).mean() * 100
axes[0,0].text(0.6, 0.8, f'Zero: {zero_pct:.1f}%', transform=axes[0,0].transAxes, fontsize=10)

# [B] Histogram log1p (nonzero)
axes[0,1].hist(np.log1p(flu_nonzero), bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0,1].set_title('Flu: log1p (nonzero only)')
axes[0,1].set_xlabel('log1p(inf_cases)')

# [C] Boxplot per WHO_REGION — xem variation giữa vùng
if 'who_region' in master_clean.columns:
    flu_log = master_clean[master_clean['inf_cases'] > 0].copy()
    flu_log['flu_log1p'] = np.log1p(flu_log['inf_cases'])
    flu_log.boxplot(column='flu_log1p', by='who_region', ax=axes[0,2], rot=30)
    axes[0,2].set_title('Flu log1p by WHO region')
    axes[0,2].set_xlabel('')
else:
    # fallback: top 10 countries boxplot
    top10_flu = master_clean.groupby('iso3')['inf_cases'].sum().nlargest(10).index.tolist()
    flu_top = master_clean[master_clean['iso3'].isin(top10_flu)].copy()
    flu_top['flu_log1p'] = np.log1p(flu_top['inf_cases'])
    flu_top.boxplot(column='flu_log1p', by='iso3', ax=axes[0,2], rot=45)
    axes[0,2].set_title('Flu log1p — top 10 countries')
    axes[0,2].set_xlabel('')

# ── DENGUE ───────────────────────────────────────────────────────────────────
dengue_cases = master_clean['dengue_total'].fillna(0)
dengue_nonzero = dengue_cases[dengue_cases > 0]

# [D] Histogram raw
axes[1,0].hist(dengue_cases, bins=80, color='coral', edgecolor='none', alpha=0.8)
axes[1,0].set_title('Dengue: raw (incl. zeros)')
axes[1,0].set_xlabel('dengue_total')
den_zero_pct = (dengue_cases == 0).mean() * 100
axes[1,0].text(0.55, 0.8, f'Zero: {den_zero_pct:.1f}%', transform=axes[1,0].transAxes, fontsize=10)

# [E] Histogram log1p (nonzero)
axes[1,1].hist(np.log1p(dengue_nonzero), bins=60, color='coral', edgecolor='none', alpha=0.8)
axes[1,1].set_title('Dengue: log1p (nonzero only)')
axes[1,1].set_xlabel('log1p(dengue_total)')

# [F] Dengue top 8 countries contribution (pie)
top8_d = master_clean.groupby('iso3')['dengue_total'].sum().nlargest(8)
others = dengue_cases.sum() - top8_d.sum()
pie_data = pd.concat([top8_d, pd.Series({'Others': others})])
axes[1,2].pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
               startangle=90, textprops={'fontsize': 8})
axes[1,2].set_title('Dengue — top 8 countries share')

plt.suptitle('SESSION 6 — Bước 3: Distribution Analysis', fontsize=14)
plt.tight_layout()
plt.show()

# ── Stats summary ─────────────────────────────────────────────────────────────
print('\n=== FLU STATS ===')
print(f'  Zero rows: {zero_pct:.1f}%')
print(f'  Nonzero rows: {len(flu_nonzero):,}')
print(f'  Mean (nonzero): {flu_nonzero.mean():.1f} | Median: {flu_nonzero.median():.0f}')
print(f'  P90: {flu_nonzero.quantile(0.9):.0f} | P99: {flu_nonzero.quantile(0.99):.0f} | Max: {flu_nonzero.max():.0f}')
print(f'  log1p mean: {np.log1p(flu_nonzero).mean():.2f} | std: {np.log1p(flu_nonzero).std():.2f}')

print('\n=== DENGUE STATS ===')
print(f'  Zero rows: {den_zero_pct:.1f}%')
print(f'  Nonzero rows: {len(dengue_nonzero):,}')
print(f'  Mean (nonzero): {dengue_nonzero.mean():.1f} | Median: {dengue_nonzero.median():.0f}')
print(f'  P90: {dengue_nonzero.quantile(0.9):.0f} | P99: {dengue_nonzero.quantile(0.99):.0f} | Max: {dengue_nonzero.max():.0f}')
print(f'  log1p mean: {np.log1p(dengue_nonzero).mean():.2f} | std: {np.log1p(dengue_nonzero).std():.2f}')

# ── Brazil dominance check ────────────────────────────────────────────────────
bra_dengue = master_clean[master_clean['iso3']=='BRA']['dengue_total'].sum()
total_dengue = master_clean['dengue_total'].sum()
print(f'\nBrazil dengue share: {bra_dengue/total_dengue*100:.1f}% ({bra_dengue/1e6:.1f}M / {total_dengue/1e6:.1f}M)')
print('→ Đây là lý do KHÔNG dùng quantile global để generate label — Brazil sẽ dominate threshold')


📌 **[6.2] Phân phối dữ liệu — Dữ liệu trông như thế nào trước khi train?**

Trước khi dạy máy tính nhận dạng dịch bệnh, cần hiểu dữ liệu đang ở dạng gì. Cell này vẽ biểu đồ phân phối để trả lời: *"Hầu hết các tuần có bao nhiêu ca? Có tuần nào bất thường không? Các nước báo cáo nhiều như nhau không?"*

---

**Cúm (Flu) — Zero: 38.4%**

38.4% các hàng trong dataset có số ca cúm = 0. Nghĩa là cứ 10 tuần thì có gần 4 tuần không ghi nhận ca nào. Điều này bình thường — cúm có mùa cao điểm (mùa đông), còn lại thì ít hoặc không có.

**Tại sao cần biết điều này?** Vì nếu mô hình không được xử lý đúng, nó sẽ "lười" — chỉ đoán "Low" (không có dịch) cho mọi trường hợp và đạt độ chính xác 38% chỉ nhờ đoán mò. Cần dùng `class_weight='balanced'` để ép mô hình học cả 3 mức.

---

**Flu: Mean 90.9 ca nhưng Median chỉ 8 ca**

Mean (trung bình) = 90.9, nhưng Median (giá trị giữa) = 8. Khoảng cách 11 lần này cho thấy: *phần lớn các nước báo rất ít ca mỗi tuần, nhưng một vài nước lớn (Mỹ, Trung Quốc, Nga) báo hàng nghìn ca và kéo trung bình lên cao.*

Hình dung: 9 người trong phòng kiếm 10 triệu/tháng, 1 người kiếm 10 tỷ → lương trung bình 1 tỷ, nhưng con số đó không đại diện cho ai cả. Median (10 triệu) mới phản ánh thực tế.

→ Đây là lý do cần baseline riêng cho từng nước, không dùng ngưỡng chung toàn cầu.

---

**Dengue — Zero: 91.0%** (nặng hơn nhiều)

91% tuần có dengue = 0. Dengue chỉ tồn tại ở các nước nhiệt đới/cận nhiệt đới (Đông Nam Á, Mỹ Latinh). Châu Âu, Bắc Mỹ, hầu hết châu Phi ôn đới → không có dengue → 0 liên tục. Với 154 nước trong 10 năm, hầu hết hàng là 0.

**Hệ quả:** Nếu model đoán "Low" cho mọi tuần → đúng 91% ngay lập tức mà không học gì. `class_weight='balanced'` là bắt buộc.

---

**Brazil chiếm 52.5% toàn bộ ca dengue thế giới**

Chỉ một nước Brazil đã chiếm hơn một nửa tổng số ca dengue toàn cầu trong 10 năm (7.1 triệu / 13.6 triệu). Đây là vấn đề nghiêm trọng nếu dùng ngưỡng toàn cầu:

*Tưởng tượng:* Nếu xếp loại "High" cho top 33% ca dengue cao nhất toàn cầu → ngưỡng đó gần như chỉ do Brazil quyết định. Một tuần Philippines có 2,000 ca (đang outbreak thật sự với đất nước họ) sẽ bị xếp "Low" vì 2,000 < ngưỡng của Brazil.

→ Giải pháp: tính baseline **riêng cho từng nước, theo từng tuần trong năm** (Endemic Channel method). Brazil so với Brazil, Philippines so với Philippines.

In [ ]:
# [6.3] BƯỚC 4 — Outlier detection: IQR + z-score + domain threshold
# Mục tiêu: phân biệt outlier thật (extreme outbreak) vs lỗi data

master_clean = pd.read_csv(MASTER_FILE)
master_clean = master_clean[~master_clean['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])]

def outlier_report(series, name, domain_max=None):
    """IQR + z-score outlier detection với domain threshold."""
    s = series.dropna()
    s_nz = s[s > 0]  # chỉ nonzero

    Q1, Q3 = s_nz.quantile(0.25), s_nz.quantile(0.75)
    IQR = Q3 - Q1
    iqr_upper = Q3 + 3 * IQR  # 3×IQR vì data rất skewed (thay vì 1.5×)

    z = (s_nz - s_nz.mean()) / s_nz.std()
    z_outliers = s_nz[z.abs() > 4]   # |z| > 4 cho skewed data

    iqr_outliers = s_nz[s_nz > iqr_upper]

    print(f'--- {name} (nonzero only) ---')
    print(f'  Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}')
    print(f'  IQR upper fence (3×IQR): {iqr_upper:.0f}')
    print(f'  IQR outliers (>{iqr_upper:.0f}): {len(iqr_outliers):,} rows ({len(iqr_outliers)/len(s_nz)*100:.1f}%)')
    print(f'  z-score |>4| outliers: {len(z_outliers):,} rows')
    if domain_max:
        domain_out = s_nz[s_nz > domain_max]
        print(f'  Domain threshold (>{domain_max:,}): {len(domain_out):,} rows')

    # Top 10 extreme values với country info
    print(f'  Top 5 values: {s_nz.nlargest(5).values}')
    print()
    return iqr_outliers, z_outliers

flu_iqr, flu_z = outlier_report(master_clean['inf_cases'], 'Flu cases', domain_max=50000)
dengue_iqr, dengue_z = outlier_report(master_clean['dengue_total'].fillna(0), 'Dengue total', domain_max=200000)

# ── Visualize outlier distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Flu: highlight outliers
flu_nz = master_clean['inf_cases'].dropna()
flu_nz = flu_nz[flu_nz > 0]
Q1f, Q3f = flu_nz.quantile(0.25), flu_nz.quantile(0.75)
fence_f = Q3f + 3 * (Q3f - Q1f)

axes[0].hist(np.log1p(flu_nz[flu_nz <= fence_f]), bins=60, color='steelblue', alpha=0.7, label='Normal')
axes[0].hist(np.log1p(flu_nz[flu_nz > fence_f]),  bins=30, color='red',       alpha=0.8, label=f'Outlier >{fence_f:.0f}')
axes[0].axvline(np.log1p(fence_f), color='red', linestyle='--', linewidth=1.5)
axes[0].set_title(f'Flu log1p: outliers (3xIQR fence={fence_f:.0f})')
axes[0].legend(); axes[0].set_xlabel('log1p(inf_cases)')

# Dengue: highlight outliers
den_nz = master_clean['dengue_total'].fillna(0)
den_nz = den_nz[den_nz > 0]
Q1d, Q3d = den_nz.quantile(0.25), den_nz.quantile(0.75)
fence_d = Q3d + 3 * (Q3d - Q1d)

axes[1].hist(np.log1p(den_nz[den_nz <= fence_d]), bins=60, color='coral', alpha=0.7, label='Normal')
axes[1].hist(np.log1p(den_nz[den_nz > fence_d]),  bins=30, color='red',   alpha=0.8, label=f'Outlier >{fence_d:.0f}')
axes[1].axvline(np.log1p(fence_d), color='red', linestyle='--', linewidth=1.5)
axes[1].set_title(f'Dengue log1p: outliers (3xIQR fence={fence_d:.0f})')
axes[1].legend(); axes[1].set_xlabel('log1p(dengue_total)')

plt.suptitle('SESSION 6 — Bước 4: Outlier Detection', fontsize=13)
plt.tight_layout(); plt.show()

print(f'\nConclusion:')
print(f'  Flu outliers (3xIQR): {len(flu_iqr):,} / {(master_clean["inf_cases"]>0).sum():,} nonzero rows ({len(flu_iqr)/(master_clean["inf_cases"]>0).sum()*100:.1f}%)')
print(f'  Dengue outliers (3xIQR): {len(dengue_iqr):,} / {(master_clean["dengue_total"].fillna(0)>0).sum():,} nonzero rows')
print('  Strategy: KEEP all — Endemic Channel auto-label extreme values as High (not errors)')


📌 **[6.3] Outlier Detection — Những con số bất thường có phải lỗi không?**

**Outlier là gì?** Khi hầu hết các tuần chỉ có vài chục ca bệnh, một tuần bỗng nhiên có 22,000 ca — đó là outlier. Câu hỏi: đó là đợt dịch thật hay ai đó nhập số liệu sai?

Cell này dùng 2 cách để phát hiện và 1 ngưỡng "thực tế" để xác nhận:
- **IQR fence (3×):** tính khoảng giữa 25%–75% của dữ liệu, rồi nhân 3 → bất kỳ giá trị nào vượt ngưỡng đó bị đánh dấu. Dùng 3× thay vì 1.5× thông thường vì dữ liệu dịch bệnh vốn rất lệch.
- **z-score >4:** cách tính khác, chặt hơn, chỉ bắt những giá trị thực sự cực đoan.
- **Domain threshold:** ngưỡng "không thể có thật" — flu >50,000 ca/tuần, dengue >200,000 ca/tuần là bất khả thi với bất kỳ nước nào.

---

**Kết quả Flu:**

Q1=3, Q3=30 — nghĩa là 75% các tuần nonzero chỉ báo ≤ 30 ca. Hầu hết nước trên thế giới có rất ít ca mỗi tuần. Fence = 111 ca.

10.7% (4,729 rows) vượt 111 ca — con số này cao nhưng đúng. Vì fence 111 rất thấp: USA, Trung Quốc, Nga báo 5,000–22,000 ca/tuần vào mùa đông là bình thường với họ, nhưng so với đa số nước khác thì là "bất thường". Top 5 giá trị cao nhất: 22,108 | 21,984 | 21,974 | 19,581 | 18,687 ca — đây là Mỹ mùa cúm 2017–2018, được ghi nhận là mùa cúm nặng nhất thập kỷ. **Domain threshold >50,000: 0 rows — không có lỗi nhập liệu.**

**Kết quả Dengue:**

Q1=61, Q3=1,192 — range rộng hơn flu nhiều vì dengue tập trung ở một số nước có qui mô dịch rất khác nhau. Fence = 4,587 ca.

8.5% (547 rows) vượt fence. Top 5: 146,906 | 144,865 | 142,057 | 140,065 | 136,886 ca/tuần — tất cả là Brazil mùa mưa 2015–2016, đợt dengue lớn nhất lịch sử Brazil. **Domain threshold >200,000: 0 rows — không có lỗi nhập liệu.**

---

**Quyết định: Giữ nguyên tất cả, không xóa, không cắt**

Không có hàng nào vượt ngưỡng "không thể có thật" → không có lỗi nhập liệu. Tất cả outlier đều là đợt dịch thật, có hồ sơ lịch sử. Với cách tiếp cận phân loại Low/Medium/High của đề tài này, những tuần outbreak cực đoan này sẽ được tự động gán nhãn High bởi bước tạo label ở SESSION sau — chính xác những gì ta muốn mô hình học.

In [ ]:
# [6.4] BƯỚC 5 — Time coverage: mỗi nước có bao nhiêu tuần dữ liệu?
# Mục tiêu: phát hiện gap, nước thiếu dữ liệu, giải thích số rows thực tế

import seaborn as sns

master_clean = pd.read_csv(MASTER_FILE)
master_clean = master_clean[~master_clean['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])]

# ── Số tuần lý thuyết vs thực tế ──────────────────────────────────────────────
years = master_clean['iso_year'].unique()
n_years = len(years)
n_countries = master_clean['iso3'].nunique()
theoretical_rows = n_countries * n_years * 52
actual_rows = len(master_clean)
print(f'Countries: {n_countries} | Years: {n_years} ({years.min()}–{years.max()})')
print(f'Theoretical rows (c × y × 52): ~{theoretical_rows:,}')
print(f'Actual rows: {actual_rows:,}')
print(f'Coverage ratio: {actual_rows/theoretical_rows*100:.1f}%')

# ── Số tuần per country ────────────────────────────────────────────────────────
weeks_per_country = master_clean.groupby('iso3').size().sort_values()
print(f'\nWeeks per country:')
print(f'  Min: {weeks_per_country.min()} ({weeks_per_country.idxmin()})')
print(f'  Max: {weeks_per_country.max()} ({weeks_per_country.idxmax()})')
print(f'  Median: {weeks_per_country.median():.0f}')
print(f'  Countries with < 400 weeks: {(weeks_per_country < 400).sum()}')
print(f'  Countries with >= 500 weeks: {(weeks_per_country >= 500).sum()}')

# ── Heatmap — top 30 flu countries ────────────────────────────────────────────
pivot = master_clean.groupby(['iso3','iso_year']).size().unstack(fill_value=0)
top30 = master_clean.groupby('iso3')['inf_cases'].sum().nlargest(30).index
pivot_top = pivot.loc[pivot.index.isin(top30)]

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

sns.heatmap(
    pivot_top,
    ax=axes[0],
    cmap='YlOrRd',
    annot=True,
    fmt='d',
    annot_kws={'size': 6},
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'weeks'}
)
axes[0].set_title('Weeks reported — top 30 flu countries\n(darker = more weeks)', fontsize=11)
axes[0].set_xlabel('')
axes[0].set_ylabel('')
axes[0].tick_params(axis='x', labelsize=8, rotation=45)
axes[0].tick_params(axis='y', labelsize=7)

# ── Histogram weeks per country ───────────────────────────────────────────────
axes[1].hist(weeks_per_country, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(weeks_per_country.median(), color='red', linestyle='--',
                label=f'Median={weeks_per_country.median():.0f}')
axes[1].set_title('Distribution: weeks per country')
axes[1].set_xlabel('Number of weeks')
axes[1].set_ylabel('Number of countries')
axes[1].legend()

plt.suptitle('SESSION 6 — Bước 5: Time Coverage Analysis', fontsize=13)
plt.tight_layout()
plt.show()

# ── Gap detection ──────────────────────────────────────────────────────────────
print('\n=== GAP DETECTION ===')
zero_year = master_clean.groupby(['iso3','iso_year'])['inf_cases'].sum()
full_zero_years = zero_year[zero_year == 0]
print(f'  (iso3, year) pairs with ALL zero flu: {len(full_zero_years):,}')
print(f'  Affected countries: {full_zero_years.index.get_level_values(0).nunique()}')
print(f'  Most affected years: {full_zero_years.index.get_level_values(1).value_counts().head(5).to_dict()}')


📌 **[6.4] Time Coverage — Mỗi nước có đủ dữ liệu theo thời gian không?**

Trước khi train mô hình, cần biết: liệu mỗi nước có dữ liệu liên tục từ 2010 đến 2019 không, hay có những "lỗ hổng" thời gian? Một nước chỉ có dữ liệu 3 năm sẽ không đủ để tạo baseline 5 năm cho Endemic Channel.

**Coverage ratio** cho biết dataset thực tế so với lý thuyết (154 nước × 10 năm × 52 tuần). Nếu tỷ lệ này thấp → nhiều nước hoặc nhiều năm bị thiếu.

**Heatmap (trái):** Mỗi ô = số tuần nước đó báo cáo trong năm đó. Màu đậm = nhiều tuần = dữ liệu tốt. Ô trắng hoặc nhạt = ít tuần hoặc không có. Nhìn vào heatmap để thấy nước nào bị gap giữa chừng.

**Histogram (phải):** Bao nhiêu nước có bao nhiêu tuần dữ liệu? Kỳ vọng: hầu hết nước có ~520 tuần (10 năm × 52 tuần). Nếu có nước chỉ 50–100 tuần → không đủ dùng cho training.

**Gap detection:** Năm nào có nhiều nước không báo flu nhất? Kỳ vọng: 2020–2021 nhiều zero vì COVID làm flu gần như biến mất. Nhưng dataset này chỉ dùng đến 2019 nên không ảnh hưởng.

In [ ]:
# [6.5] BƯỚC 6 — CV split logic: walk-forward, không data leakage
# Mục tiêu: visualize rõ train/val split theo từng fold, confirm không nhìn tương lai

master_clean = pd.read_csv(MASTER_FILE)
master_clean = master_clean[~master_clean['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])]

# ── Walk-forward CV scheme ─────────────────────────────────────────────────────
# Train: 2010 → (val_year - 1) | Val: val_year | Test: 2022 (held-out)
val_years = [2014, 2015, 2016, 2017, 2018, 2019]
all_years = sorted(master_clean['iso_year'].unique())

print('Walk-forward CV — 6 folds:')
print(f'{"Fold":<6} {"Train years":<25} {"Val year":<10} {"Train rows":>12} {"Val rows":>10}')
print('-' * 70)

fold_info = []
for fold, val_year in enumerate(val_years, 1):
    train_years = [y for y in all_years if y < val_year]
    train_mask = master_clean['iso_year'].isin(train_years)
    val_mask   = master_clean['iso_year'] == val_year
    n_train = train_mask.sum()
    n_val   = val_mask.sum()
    train_str = f'{min(train_years)}–{max(train_years)}'
    print(f'{fold:<6} {train_str:<25} {val_year:<10} {n_train:>12,} {n_val:>10,}')
    fold_info.append((fold, train_years, val_year, n_train, n_val))

print()
print(f'Test set (held-out, never touched): 2022')
print(f'Excluded: 2020–2021 (COVID NPI bias — flu cases dropped ~99% artificially)')

# ── Visualize timeline ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

colors = {'train': '#4C9BE8', 'val': '#F28C28', 'test': '#2ECC71', 'excluded': '#CCCCCC'}

for fold, train_years, val_year, _, _ in fold_info:
    y = fold
    for yr in train_years:
        ax.barh(y, 1, left=yr, color=colors['train'], alpha=0.6, height=0.6)
    ax.barh(y, 1, left=val_year, color=colors['val'], alpha=0.9, height=0.6)
    ax.text(val_year + 0.5, y, str(val_year), ha='center', va='center', fontsize=7, color='white', fontweight='bold')

# Test + excluded
ax.barh(0.0, 1, left=2022, color=colors['test'],     alpha=0.9, height=0.4)
ax.barh(0.0, 1, left=2023, color=colors['test'],     alpha=0.9, height=0.4)
for yr in [2020, 2021]:
    ax.barh(0.0, 1, left=yr, color=colors['excluded'], alpha=0.7, height=0.4)

ax.text(2022.5, 0.0, 'Test 2022', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
ax.text(2020.5, 0.35, 'excluded', ha='center', va='bottom', fontsize=6, color='gray')

ax.set_yticks(list(range(1, 7)) + [0])
ax.set_yticklabels([f'Fold {f}' for f in range(1, 7)] + ['Final eval'])
ax.set_xlabel('Year')
ax.set_xlim(2009, 2024)
ax.set_title('Walk-forward Cross-Validation — 6 folds (no data leakage)', fontsize=12)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors['train'],    alpha=0.6, label='Train'),
    Patch(facecolor=colors['val'],      alpha=0.9, label='Validation (val_year)'),
    Patch(facecolor=colors['test'],     alpha=0.9, label='Test (held-out 2022)'),
    Patch(facecolor=colors['excluded'], alpha=0.7, label='Excluded (COVID 2020–2021)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

# ── Confirm no leakage ────────────────────────────────────────────────────────
print('\nLeakage check:')
for fold, train_years, val_year, _, _ in fold_info:
    overlap = set(train_years) & {val_year}
    status = 'OK' if not overlap else 'LEAK!'
    print(f'  Fold {fold}: train max year={max(train_years)}, val={val_year} → {status}')


📌 **[6.5]** Walk-forward cross-validation — cách chia train/validation đúng cho dữ liệu thời gian.

Với dữ liệu thông thường (ảnh, văn bản...), có thể xáo trộn ngẫu nhiên rồi chia 80/20. Nhưng với dữ liệu theo thời gian như dịch bệnh, làm vậy sẽ bị "data leakage" — tức model được nhìn dữ liệu tương lai để dự đoán quá khứ, kết quả trên giấy rất đẹp nhưng khi dùng thật thì sai.

Walk-forward CV giải quyết bằng cách: mỗi fold chỉ train trên quá khứ, validate trên 1 năm tiếp theo chưa từng thấy. 6 folds = 6 lần thử nghiệm độc lập trên 6 năm khác nhau (2014–2019), bao gồm cả năm bình thường lẫn năm có dịch lớn.

2020–2021 bị loại vì COVID làm ca cúm giảm ~99% do giãn cách xã hội — nếu train trên dữ liệu này, model sẽ học pattern sai (không phải mùa dịch thật). 2022 là test set cuối cùng, hoàn toàn không được nhìn trong suốt quá trình train.

In [ ]:
# [6.6] BƯỚC 7 — Sanity check dịch tễ: top countries, peak season
# Mục tiêu: xác nhận data khớp kiến thức thực tế về dịch bệnh

master_clean = pd.read_csv(MASTER_FILE)
master_clean = master_clean[~master_clean['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── [A] Top 10 flu countries ──────────────────────────────────────────────────
top10_flu = master_clean.groupby('iso3')['inf_cases'].sum().nlargest(10)
axes[0,0].barh(top10_flu.index[::-1], top10_flu.values[::-1] / 1e6, color='steelblue')
axes[0,0].set_xlabel('Total cases (millions)')
axes[0,0].set_title('Top 10 flu countries (2010–2019)')

# ── [B] Flu seasonality — average cases by ISO week (global) ─────────────────
flu_by_week = master_clean.groupby('iso_week')['inf_cases'].mean()
axes[0,1].bar(flu_by_week.index, flu_by_week.values, color='steelblue', alpha=0.8)
axes[0,1].set_xlabel('ISO week')
axes[0,1].set_ylabel('Avg cases (global)')
axes[0,1].set_title('Flu seasonality — global avg by week')
axes[0,1].axvspan(1, 10,  alpha=0.08, color='blue', label='NH winter (w1-10)')
axes[0,1].axvspan(40, 52, alpha=0.08, color='blue')
axes[0,1].axvspan(20, 35, alpha=0.08, color='red',  label='SH winter (w20-35)')
axes[0,1].legend(fontsize=8)

# ── [C] Top 10 dengue countries ───────────────────────────────────────────────
top10_den = master_clean.groupby('iso3')['dengue_total'].sum().nlargest(10)
axes[1,0].barh(top10_den.index[::-1], top10_den.values[::-1] / 1e6, color='coral')
axes[1,0].set_xlabel('Total cases (millions)')
axes[1,0].set_title('Top 10 dengue countries (2010–2019)')

# ── [D] Dengue seasonality — average cases by ISO week (global) ──────────────
den_by_week = master_clean.groupby('iso_week')['dengue_total'].mean()
axes[1,1].bar(den_by_week.index, den_by_week.values, color='coral', alpha=0.8)
axes[1,1].set_xlabel('ISO week')
axes[1,1].set_ylabel('Avg cases (global)')
axes[1,1].set_title('Dengue seasonality — global avg by week')
axes[1,1].axvspan(1, 15,  alpha=0.08, color='red', label='Rainy season NH (Jan-Apr)')
axes[1,1].axvspan(40, 52, alpha=0.08, color='red')
axes[1,1].legend(fontsize=8)

plt.suptitle('SESSION 6 — Bước 7: Sanity Check Dịch Tễ', fontsize=13)
plt.tight_layout()
plt.show()

# ── Text summary ──────────────────────────────────────────────────────────────
print('=== TOP 5 FLU COUNTRIES ===')
for iso, val in top10_flu.head(5).items():
    print(f'  {iso}: {val/1e6:.2f}M cases')

print('\n=== TOP 5 DENGUE COUNTRIES ===')
for iso, val in top10_den.head(5).items():
    print(f'  {iso}: {val/1e6:.2f}M cases')

print('\n=== FLU PEAK WEEKS (global avg) ===')
top5_weeks = flu_by_week.nlargest(5)
print(f'  {list(top5_weeks.index)} → avg {top5_weeks.values.mean():.1f} cases/week')

print('\n=== DENGUE PEAK WEEKS (global avg) ===')
top5_den_weeks = den_by_week.nlargest(5)
print(f'  {list(top5_den_weeks.index)} → avg {top5_den_weeks.values.mean():.1f} cases/week')


📌 **[6.6]** Sanity check — kiểm tra xem dữ liệu có "hợp lý về mặt y tế" không, trước khi đưa vào train.

Đây là bước quan trọng thường bị bỏ qua: một pipeline có thể chạy không lỗi nhưng vẫn cho ra kết quả sai nếu dữ liệu bị lệch về mặt thực tế. Ví dụ: nếu top flu country là Chad thay vì Mỹ/Trung Quốc, hoặc peak cúm lại rơi vào mùa hè — đó là dấu hiệu data bị sai.

**[A] Top 10 flu countries:** kỳ vọng thấy Mỹ, Trung Quốc, Nga, châu Âu — các nước có hệ thống giám sát tốt và dân số lớn.

**[B] Flu seasonality theo tuần:** kỳ vọng thấy 2 đỉnh — tuần 1–10 (mùa đông Bắc bán cầu: Mỹ, châu Âu, Trung Quốc) và tuần 20–35 (mùa đông Nam bán cầu: Úc, Nam Mỹ). Nếu chỉ có 1 đỉnh hoặc đỉnh sai vị trí → data có vấn đề.

**[C] Top 10 dengue countries:** kỳ vọng Brazil chiếm đầu, tiếp theo là các nước Đông Nam Á (Philippines, Thailand, Indonesia) và Mỹ Latinh.

**[D] Dengue seasonality theo tuần:** kỳ vọng đỉnh vào đầu năm (tuần 1–15) và cuối năm (tuần 40–52) — tương ứng mùa mưa ở Brazil và Đông Nam Á khi muỗi sinh sản mạnh.

In [ ]:
# [6.7] BƯỚC 8 — Endemic Channel preview: thử generate label cho 1 nước
# Mục tiêu: xác nhận formula Bortman 1999 hoạt động đúng trước khi làm toàn bộ SESSION 9

master_clean = pd.read_csv(MASTER_FILE)
master_clean = master_clean[~master_clean['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])]

def endemic_channel_labels(df, iso3_target, disease_col, min_years=5):
    """
    Generate Low/Medium/High labels per (iso3, iso_week) using Bortman 1999.
    baseline = mean(cases, last 5 years, same ISO week)
    sd       = std(cases, last 5 years, same ISO week)
    Low    : cases < baseline
    Medium : baseline <= cases < baseline + 2*sd
    High   : cases >= baseline + 2*sd
    """
    country_df = df[df['iso3'] == iso3_target].copy().sort_values(['iso_year','iso_week'])
    country_df['label'] = None
    country_df['baseline'] = np.nan
    country_df['upper_bound'] = np.nan

    for idx, row in country_df.iterrows():
        yr, wk = row['iso_year'], row['iso_week']
        # Lấy cùng ISO week, 5 năm trước
        hist = country_df[
            (country_df['iso_week'] == wk) &
            (country_df['iso_year'] < yr) &
            (country_df['iso_year'] >= yr - 5)
        ][disease_col]

        if len(hist) < min_years:
            continue  # Không đủ lịch sử

        baseline = hist.mean()
        sd = hist.std()
        upper = baseline + 2 * sd
        cases = row[disease_col] if not pd.isna(row[disease_col]) else 0

        country_df.loc[idx, 'baseline'] = baseline
        country_df.loc[idx, 'upper_bound'] = upper
        if cases < baseline:
            country_df.loc[idx, 'label'] = 'Low'
        elif cases < upper:
            country_df.loc[idx, 'label'] = 'Medium'
        else:
            country_df.loc[idx, 'label'] = 'High'

    return country_df

# ── Preview: USA flu ───────────────────────────────────────────────────────────
usa = endemic_channel_labels(master_clean, 'USA', 'inf_cases')
usa_labeled = usa.dropna(subset=['label'])

print('=== USA FLU — Endemic Channel Label Preview ===')
print(f'  Labeled rows: {len(usa_labeled)} / {len(usa)} total')
print(f'  Label distribution:')
print(usa_labeled['label'].value_counts().to_string())
print(f'\n  Label %:')
print((usa_labeled['label'].value_counts(normalize=True)*100).round(1).to_string())

# ── Preview: BRA dengue ────────────────────────────────────────────────────────
bra = endemic_channel_labels(master_clean, 'BRA', 'dengue_total')
bra_labeled = bra.dropna(subset=['label'])

print('\n=== BRAZIL DENGUE — Endemic Channel Label Preview ===')
print(f'  Labeled rows: {len(bra_labeled)} / {len(bra)} total')
print(f'  Label distribution:')
print(bra_labeled['label'].value_counts().to_string())

# ── Visualize: USA flu Endemic Channel 2015–2019 ───────────────────────────────
usa_plot = usa_labeled[usa_labeled['iso_year'] >= 2015].copy()
usa_plot['date_approx'] = pd.to_datetime(
    usa_plot['iso_year'].astype(str) + '-W' + usa_plot['iso_week'].astype(str).str.zfill(2) + '-1',
    format='%G-W%V-%u', errors='coerce'
)

color_map = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

for ax, (iso3, disease_col, df_plot, title) in zip(axes, [
    ('USA', 'inf_cases',    usa_labeled[usa_labeled['iso_year'] >= 2015].copy(), 'USA Flu — Endemic Channel Labels (2015–2019)'),
    ('BRA', 'dengue_total', bra_labeled[bra_labeled['iso_year'] >= 2015].copy(), 'Brazil Dengue — Endemic Channel Labels (2015–2019)'),
]):
    df_plot = df_plot.copy()
    df_plot['date_approx'] = pd.to_datetime(
        df_plot['iso_year'].astype(str) + '-W' + df_plot['iso_week'].astype(str).str.zfill(2) + '-1',
        format='%G-W%V-%u', errors='coerce'
    )
    df_plot = df_plot.dropna(subset=['date_approx']).sort_values('date_approx')

    ax.fill_between(df_plot['date_approx'], df_plot['baseline'], df_plot['upper_bound'],
                    alpha=0.2, color='orange', label='Endemic zone (baseline → baseline+2σ)')
    ax.plot(df_plot['date_approx'], df_plot['baseline'], 'k--', linewidth=1, label='Baseline')

    for label, color in color_map.items():
        mask = df_plot['label'] == label
        ax.scatter(df_plot[mask]['date_approx'], df_plot[mask][disease_col],
                   color=color, s=15, label=label, zorder=3)

    ax.set_title(title)
    ax.set_ylabel('Cases')
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('SESSION 6 — Bước 8: Endemic Channel Preview', fontsize=13)
plt.tight_layout()
plt.show()


📌 **[6.7]** Thử nghiệm công thức tạo nhãn Endemic Channel (Bortman 1999) trên USA (flu) và Brazil (dengue) trước khi áp dụng toàn bộ 154 nước ở SESSION 9.

Với mỗi nước và mỗi tuần trong năm, lấy 5 năm lịch sử cùng tuần đó để tính baseline (trung bình) và độ lệch chuẩn. Sau đó so ca hiện tại: dưới baseline → Low, từ baseline đến baseline+2σ → Medium, vượt baseline+2σ → High. Vùng cam trong biểu đồ là "vùng bình thường" (endemic zone).

Kết quả USA flu: Low 50% / Medium 36% / High 14% — hợp lý, nửa số tuần là mùa không có dịch, chỉ 14% thực sự vượt ngưỡng bùng phát. Cụm đỏ dày đặc năm 2018 khớp với mùa H3N2 nặng nhất thập kỷ tại Mỹ.

Kết quả Brazil dengue: Medium 45% / High 35% / Low 20% — High chiếm nhiều hơn vì Brazil là nước endemic dengue, bùng phát xảy ra thường xuyên. Năm 2015–2016 nhiều điểm đỏ khớp với đợt dengue lịch sử Brazil. Điều đáng chú ý: sau năm dịch lớn, baseline tự tăng và endemic zone mở rộng theo — hệ thống tự điều chỉnh, không dùng ngưỡng cố định mãi mãi.

262 rows USA và 261 rows Brazil không được label vì chưa đủ 5 năm lịch sử (2010–2014) — đúng như thiết kế, sẽ loại ra khi train. Formula xác nhận hoạt động đúng — sẵn sàng cho SESSION 9.

# 📊 SESSION 7 — CORRELATION & LAG ANALYSIS
> **Input:** master_weekly_2010_2019.csv
> **Mục tiêu:** Xác nhận lag time weather→disease bằng cross-correlation
> **Skip nếu:** Đã chạy và có kết quả (không có output file — session này chỉ visualize)

In [ ]:
# [7.0] RESTART CELL — load master data
master = pd.read_csv(MASTER_FILE)
print(f"Loaded master: {master.shape}")

flu_train = master[
    (master['iso_year'] >= TRAIN_START) &
    (master['iso_year'] <= TRAIN_END) &
    (master[TARGET_FLU].notna())
].copy()

dengue_train = master[
    (master['iso_year'] >= TRAIN_START) &
    (master['iso_year'] <= TRAIN_END) &
    (master[TARGET_DENGUE] > 0)
].copy()

print(f"flu_train: {flu_train.shape}")
print(f"dengue_train: {dengue_train.shape}")

📌 **[7.0]** Load lại data từ CSV thay vì dùng biến từ session trước — đảm bảo session independence. 
Filter `dengue_log1p > 0` loại bỏ các tuần không có dịch, giữ lại signal thực sự.

In [ ]:
# [7.1] Heatmap correlation: disease x disease
disease_cols = [c for c in ['inf_cases', 'rsv_cases', 'dengue_total', 'dengue_log1p']
                if c in master.columns]
corr_disease = master[disease_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_disease, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Disease cross-correlation (lag=0)')
plt.tight_layout()
plt.show()

print(f'Columns used: {disease_cols}')
print(corr_disease.round(3).to_string())

📌 **[7.1]** Heatmap tương quan giữa các bệnh với nhau. Cúm và RSV có tương quan 0.16 — hợp lý vì cả hai đều là bệnh hô hấp xảy ra cùng mùa lạnh. Cúm và dengue gần bằng 0 (0.002) — hoàn toàn đúng: hai bệnh khác cơ chế, khác khí hậu, khác địa lý. `dengue_log1p` tương quan với `dengue_total` chỉ 0.35 thay vì ~1.0 vì log1p nén mạnh các giá trị Brazil cực đoan — bằng chứng thực nghiệm cho quyết định không dùng `dengue_log1p` làm feature.

In [ ]:
# [7.2] Heatmap: weather x influenza (lag=0)
weather_inf_corr = flu_train[WEATHER_VARS + [TARGET_FLU]].corr()[[TARGET_FLU]].drop(TARGET_FLU)
weather_inf_corr = weather_inf_corr.sort_values(TARGET_FLU, ascending=False)

fig, ax = plt.subplots(figsize=(4, 8))
sns.heatmap(weather_inf_corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Weather x Influenza\nCorrelation (lag=0)', fontsize=12)
plt.tight_layout()
plt.show()

📌 **[7.2]** Tương quan tức thì (lag=0) giữa thời tiết và cúm — tất cả đều rất yếu (-0.17 đến 0.06). Nhiệt độ (`temp_c`, `temp_min_c`, `temp_max_c`) ở -0.17: lạnh hơn thì có xu hướng nhiều cúm hơn, nhưng tác động không xảy ra ngay. Humidity gần 0 ở lag=0. Điều này cho thấy thời tiết ảnh hưởng đến dịch bệnh qua cơ chế trễ — cần cross-correlation theo lag để tìm đúng độ trễ.

In [ ]:
# [7.3] Cross-correlation: influenza vs temp_c and humidity_pct

def lag_corr(series_x, series_y, max_lag=12):
    """Cross-correlation between x (lagged) and y."""
    return [series_x.shift(lag).corr(series_y) for lag in range(0, max_lag + 1)]

flu_weekly = flu_train.groupby(['iso_year', 'iso_week'])[[TARGET_FLU, 'temp_c', 'humidity_pct']].mean()
lags = list(range(0, 9))
corr_temp  = lag_corr(flu_weekly['temp_c'], flu_weekly[TARGET_FLU], max_lag=8)
corr_humid = lag_corr(flu_weekly['humidity_pct'], flu_weekly[TARGET_FLU], max_lag=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, corrs, var in zip(axes, [corr_temp, corr_humid], ['temp_c', 'humidity_pct']):
    bars = ax.bar(lags, corrs, color=['#e74c3c' if c < 0 else '#3498db' for c in corrs])
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'CCF: {var} → inf_cases')
    ax.set_xticks(lags)
    for bar, c in zip(bars, corrs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{c:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

📌 **[7.3]** Cross-correlation xác nhận độ trễ (lag) tối ưu giữa thời tiết và cúm. Nhiệt độ có tương quan âm (-0.63 đến -0.73) ở tất cả lag 0–8, mạnh nhất ở lag 4–5: nhiệt độ giảm hôm nay → 4–5 tuần sau ca cúm tăng, khớp với thời gian ủ bệnh và lan truyền cộng đồng. Humidity dương và tăng liên tục đến lag 8 (0.65), chưa thấy đỉnh — cần mở rộng thêm. Quyết định: **temp lag=4, humidity lag=8**.

In [ ]:
# [7.4] Cross-correlation: dengue vs precip_mm and temp_c

dengue_weekly = dengue_train.groupby(['iso_year','iso_week'])[[TARGET_DENGUE,'precip_mm','temp_c']].mean()
lags_d = list(range(0, 17))
corr_precip = lag_corr(dengue_weekly['precip_mm'], dengue_weekly[TARGET_DENGUE], max_lag=16)
corr_temp_d = lag_corr(dengue_weekly['temp_c'], dengue_weekly[TARGET_DENGUE], max_lag=16)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, corrs, var in zip(axes, [corr_precip, corr_temp_d], ['precip_mm', 'temp_c']):
    bars = ax.bar(lags_d, corrs, color=['#e74c3c' if c < 0 else '#27ae60' for c in corrs])
    ax.axhline(0, color='black', lw=0.8)
    ax.axvspan(6, 14, alpha=0.12, color='green', label='LAG_DENGUE zone')
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'CCF: {var} → dengue_log1p')
    ax.set_xticks(lags_d[::2])
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

📌 **[7.4]** Cross-correlation mưa và nhiệt độ với dengue — kết quả âm ở tất cả lag, bề ngoài ngược lý (mưa nhiều → ít dengue?). Thực ra đây là hiệu ứng của global average: Brazil chiếm 52.5% ca dengue và có peak tháng 1–4 (mùa hè Nam bán cầu), trong khi lúc đó toàn cầu đang đông lạnh, ít mưa → trung bình toàn cầu ra tương quan âm giả tạo. CCF global không đủ tin cậy cho dengue, nên lag được quyết định dựa vào sinh học vector.

Vòng đời Aedes aegypti: Trứng (2–7 ngày) → Ấu trùng (5–7 ngày nước đọng) → Nhộng (2 ngày) → Muỗi trưởng thành ủ virus (8–12 ngày) → Người ủ bệnh (4–10 ngày). Tổng độ trễ từ mưa đến ca bệnh: **3–5 tuần**.

**Lag dengue chốt: temp=0w, humidity=2w, solar=4w, precip=2w (thêm precip_lag4 làm feature phụ).** Nhiệt độ ảnh hưởng trực tiếp đến hoạt động muỗi nên lag=0; mưa cần thời gian tạo nơi sinh sản và hoàn thành vòng đời nên lag=2–4.

In [ ]:
# [7.5] Summary heatmap: top weather vars x disease targets x lag

top_vars = ['temp_c', 'humidity_pct', 'precip_mm', 'solar_wm2', 'dewpoint_c']
lag_points = [0, 2, 4, 8, 12]
disease_map = {'inf_cases': flu_train, 'dengue_log1p': dengue_train}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (dis_name, df) in zip(axes, disease_map.items()):
    weekly = df.groupby(['iso_year','iso_week'])[top_vars + [dis_name]].mean()
    corr_mat = []
    for var in top_vars:
        row = [weekly[var].shift(lag).corr(weekly[dis_name]) for lag in lag_points]
        corr_mat.append(row)
    corr_df = pd.DataFrame(corr_mat, index=top_vars,
                            columns=[f'lag{l}w' for l in lag_points])
    sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdYlBu_r',
                center=0, vmin=-0.6, vmax=0.6, ax=ax, linewidths=0.4)
    ax.set_title(f'Weather x {dis_name}', fontsize=12)

plt.suptitle('Summary Weather Lag Correlation', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[7.5]** Bảng tổng kết tương quan lag giữa các biến thời tiết và 2 bệnh. Mỗi ô là hệ số Pearson r — đỏ đậm = tương quan dương mạnh, xanh đậm = tương quan âm mạnh.

Flu: `solar_wm2` có signal mạnh nhất (-0.76 ở lag8w) — ít ánh sáng mặt trời tức là mùa đông, đúng thời điểm cúm bùng phát. `temp_c` peak -0.73 ở lag4w. `dewpoint_c` -0.72 ở lag2–4w. `humidity_pct` 0.65 ở lag8w. **Flu lags chốt: temp=4w, solar=8w, dewpoint=2w, humidity=8w.**

Dengue: `temp_c` và `dewpoint_c` dương và flat qua tất cả lag (~0.49–0.52) — endemic ở vùng nhiệt đới nên luôn warm/humid. `solar_wm2` gần 0 — bỏ. `precip_mm` mạnh nhất lag0w nhưng sinh học vector cho thấy lag 2–4w hợp lý hơn. **Dengue lags chốt: temp=0, dewpoint=2w, humidity=2w, precip=2w.**

---

# 🔧 SESSION 8 — FEATURE ENGINEERING
> **Input:** `master_weekly_2010_2019.csv`
> **Output:** `features_flu.csv`, `features_dengue.csv` — sẵn sàng cho training
>
> Xây dựng feature matrix từ lag đã chốt ở SESSION 7 + autoregressive lags + seasonal features + climate zone.

In [ ]:
# [8.0] RESTART — load + fix Russia aggregate
master = pd.read_csv(MASTER_FILE)
master = master[~master['iso3'].isin([
    'BLM','BRB','BMU','GLP','GUF','DMA','MTQ','MLT','MDV',
    'MAF','KNA','TCA','SGP','X12','X10','XKX','X09','X11'
])].copy()

# Fix Russia: aggregate sub-national rows → 1 row per (iso3, iso_year, iso_week)
numeric_cols = master.select_dtypes(include='number').columns.tolist()
master = master.groupby(['iso3','iso_year','iso_week'])[numeric_cols].mean().reset_index()

print(f'After drop 18 + RUS aggregate: {master.shape}')
print(f'RUS rows: {len(master[master["iso3"]=="RUS"])} (expect ~520)')
print(f'Columns: {list(master.columns)}')


📌 **[8.0]** Load data và xử lý 2 vấn đề: drop 18 nước không có ERA5, aggregate tất cả duplicate (iso3, year, week) về 1 hàng bằng mean. Russia giảm từ 1,131 xuống 521 rows (đúng ~520 tuần/10 năm). Tổng dataset giảm từ 71,717 xuống 59,270 — phần còn lại ngoài Russia là các nước khác trong FluNet cũng có sub-national entries chưa được gộp. Sau bước này mỗi (nước, năm, tuần) là duy nhất, sẵn sàng tạo lag features.

In [ ]:
# [8.1] Tạo weather lag features — từ lag chốt ở SESSION 7
# Flu: temp lag4, solar lag8, dewpoint lag2, humidity lag8
# Dengue: temp lag0, dewpoint lag2, humidity lag2, precip lag2

master = master.sort_values(['iso3','iso_year','iso_week'])

def add_lag(df, col, lag, new_name=None):
    """Shift col by lag weeks within each country."""
    name = new_name or f'{col}_lag{lag}w'
    df[name] = df.groupby('iso3')[col].shift(lag)
    return df

# ── Flu weather lags ──────────────────────────────────────────────────────────
for col, lag in [('temp_c', 4), ('solar_wm2', 8), ('dewpoint_c', 2), ('humidity_pct', 8)]:
    master = add_lag(master, col, lag)

# ── Dengue weather lags ───────────────────────────────────────────────────────
for col, lag in [('temp_c', 0), ('dewpoint_c', 2), ('humidity_pct', 2), ('precip_mm', 2)]:
    name = f'{col}_den_lag{lag}w' if lag > 0 else f'{col}_den'
    master = add_lag(master, col, lag, new_name=name)

print('Weather lag features added:')
lag_cols = [c for c in master.columns if '_lag' in c or '_den' in c]
print(f'  {lag_cols}')
print(f'Master shape: {master.shape}')


📌 **[8.1]** Tạo weather lag features từ quyết định SESSION 7. Shift theo từng nước (groupby iso3) — không shift toàn bảng, tránh lấy nhầm dữ liệu nước khác.

- **Flu:** temp lag4w, solar lag8w, dewpoint lag2w, humidity lag8w
- **Dengue:** temp lag0, dewpoint lag2w, humidity lag2w, precip lag2w (lag ngắn hơn vì vòng đời muỗi ~3–5 tuần)

Shape: 25 → 33 columns (thêm đúng 8 features).

In [ ]:
# [8.2] Autoregressive lags + seasonal features
# AR lags: cases[-1], cases[-2], cases[-4] — tín hiệu tự hồi quy
# Seasonal: sin/cos của iso_week để encode mùa liên tục

import numpy as np

# ── AR lags cho flu ────────────────────────────────────────────────────────────
for lag in [1, 2, 4]:
    master = add_lag(master, 'inf_cases', lag)

# ── AR lags cho dengue ────────────────────────────────────────────────────────
for lag in [1, 2, 4]:
    master = add_lag(master, 'dengue_total', lag)

# ── Seasonal encoding: sin/cos (iso_week) ─────────────────────────────────────
master['week_sin'] = np.sin(2 * np.pi * master['iso_week'] / 52)
master['week_cos'] = np.cos(2 * np.pi * master['iso_week'] / 52)

# ── Climate zone (thay cho who_region_enc chưa có) ────────────────────────────
# Tạo từ latitude — dùng approximate centroid lookup
# Simple encoding: tropical (<23.5°), temperate (23.5–60°), cold (>60°)
# Sẽ merge với climate_zone_enc sau — tạm thời skip

print('AR lag features + seasonal:')
ar_cols = [c for c in master.columns if 'inf_cases_lag' in c or 'dengue_total_lag' in c]
print(f'  AR: {ar_cols}')
print(f'  Seasonal: week_sin, week_cos')
print(f'Master shape: {master.shape}')

# Quick check: bao nhiêu NaN sau khi shift?
lag_feature_cols = [c for c in master.columns if '_lag' in c]
nan_pct = master[lag_feature_cols].isna().mean() * 100
print(f'\nNaN % per lag feature (top 5 highest):')
print(nan_pct.nlargest(5).round(1).to_string())


📌 **[8.2]** Thêm 2 nhóm features:

- **AR lags:** ca bệnh lag1w/2w/4w cho cả flu và dengue — dịch có tính quán tính, tuần trước nhiều ca thì tuần này thường cũng cao.
- **Seasonal:** `week_sin` và `week_cos` thay vì số tuần thô — giúp model hiểu tuần 1 và tuần 52 "gần nhau" về mùa, không phải cách xa 51 tuần.

NaN cao nhất 2.2% (lag8w) — mỗi nước mất 8 tuần đầu khi shift. Sẽ dropna ở bước tiếp theo. Shape: 33 → 41 columns.

In [ ]:
# [8.3] Build final feature matrices và save CSV

FLU_WEATHER_LAGS   = ['temp_c_lag4w', 'solar_wm2_lag8w', 'dewpoint_c_lag2w', 'humidity_pct_lag8w']
DENGUE_WEATHER_LAGS = ['temp_c_den', 'dewpoint_c_den_lag2w', 'humidity_pct_den_lag2w', 'precip_mm_den_lag2w']
FLU_AR_LAGS        = ['inf_cases_lag1w', 'inf_cases_lag2w', 'inf_cases_lag4w']
DENGUE_AR_LAGS     = ['dengue_total_lag1w', 'dengue_total_lag2w', 'dengue_total_lag4w']
SEASONAL           = ['week_sin', 'week_cos']
META               = ['iso3', 'iso_year', 'iso_week']

# ── Flu feature matrix ────────────────────────────────────────────────────────
flu_cols = META + FLU_WEATHER_LAGS + FLU_AR_LAGS + SEASONAL + ['inf_cases']
flu_df = master[flu_cols].dropna()
# Chỉ lấy train years
flu_df = flu_df[(flu_df['iso_year'] >= TRAIN_START) & (flu_df['iso_year'] <= TRAIN_END)]

# ── Dengue feature matrix ─────────────────────────────────────────────────────
dengue_cols = META + DENGUE_WEATHER_LAGS + DENGUE_AR_LAGS + SEASONAL + ['dengue_total']
dengue_df = master[dengue_cols].dropna()
dengue_df = dengue_df[
    (dengue_df['iso_year'] >= TRAIN_START) &
    (dengue_df['iso_year'] <= TRAIN_END) &
    (dengue_df['dengue_total'] >= 0)
]

print(f'Flu feature matrix:    {flu_df.shape}')
print(f'  Countries: {flu_df["iso3"].nunique()}')
print(f'  Features: {[c for c in flu_df.columns if c not in META + ["inf_cases"]]}')

print(f'\nDengue feature matrix: {dengue_df.shape}')
print(f'  Countries: {dengue_df["iso3"].nunique()}')
print(f'  Features: {[c for c in dengue_df.columns if c not in META + ["dengue_total"]]}')

# Save
FLU_FEATURES_FILE    = PROCESSED / 'features_flu_v3.csv'
DENGUE_FEATURES_FILE = PROCESSED / 'features_dengue_v3.csv'
flu_df.to_csv(FLU_FEATURES_FILE, index=False)
dengue_df.to_csv(DENGUE_FEATURES_FILE, index=False)
print(f'\nSaved: {FLU_FEATURES_FILE.name}')
print(f'Saved: {DENGUE_FEATURES_FILE.name}')


📌 **[8.3]** Ghép tất cả features thành 2 file CSV riêng:

- **Flu:** 57,739 rows × 9 features (4 weather lag + 3 AR lag + 2 seasonal), 149 nước
- **Dengue:** 58,448 rows × 9 features, 152 nước — nhiều hơn flu vì lag ngắn hơn, ít bị drop NaN

Giữ cả rows `dengue_total = 0` vì classification cần label Low cho tuần không có dịch. Output: `features_flu_v3.csv` và `features_dengue_v3.csv` — input cho SESSION 9.

In [ ]:
# [8.4] Thêm who_region_enc + quarter vào features_v3 (improvements đã proven từ approach cũ)

# WHO regions mapping (ISO3 → AFR/AMR/EMR/EUR/SEAR/WPR)
WHO_REGIONS = {
    # AFR — African Region (47)
    'AFR': ['DZA','AGO','BEN','BWA','BFA','BDI','CPV','CMR','CAF','TCD','COM','COG','COD','CIV',
            'GNQ','ERI','SWZ','ETH','GAB','GMB','GHA','GIN','GNB','KEN','LSO','LBR','MDG','MWI',
            'MLI','MRT','MUS','MOZ','NAM','NER','NGA','RWA','STP','SEN','SYC','SLE','ZAF','SSD',
            'TGO','UGA','TZA','ZMB','ZWE'],
    # AMR — Region of the Americas (35)
    'AMR': ['ATG','ARG','BHS','BRB','BLZ','BOL','BRA','CAN','CHL','COL','CRI','CUB','DMA','DOM',
            'ECU','SLV','GRD','GTM','GUY','HTI','HND','JAM','MEX','NIC','PAN','PRY','PER','KNA',
            'LCA','VCT','SUR','TTO','USA','URY','VEN','ABW','AIA','CYM'],
    # EMR — Eastern Mediterranean Region (21)
    'EMR': ['AFG','BHR','DJI','EGY','IRN','IRQ','JOR','KWT','LBN','LBY','MAR','OMN','PAK','PSE',
            'QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM'],
    # EUR — European Region (53)
    'EUR': ['ALB','AND','ARM','AUT','AZE','BLR','BEL','BIH','BGR','HRV','CYP','CZE','DNK','EST',
            'FIN','FRA','GEO','DEU','GRC','HUN','ISL','IRL','ISR','ITA','KAZ','KGZ','LVA','LTU',
            'LUX','MLT','MCO','MNE','NLD','MKD','NOR','POL','PRT','MDA','ROU','RUS','SMR','SRB',
            'SVK','SVN','ESP','SWE','CHE','TJK','TUR','TKM','UKR','GBR','UZB'],
    # SEAR — South-East Asia Region (11)
    'SEAR': ['BGD','BTN','PRK','IND','IDN','MDV','MMR','NPL','LKA','THA','TLS'],
    # WPR — Western Pacific Region (27)
    'WPR': ['AUS','BRN','KHM','CHN','COK','FJI','JPN','KIR','LAO','MYS','MHL','FSM','MNG','NRU',
            'NZL','NIU','PLW','PNG','PHL','KOR','WSM','SGP','SLB','TON','TUV','VUT','VNM','HKG','MAC','NCL'],
}

# Reverse map: iso3 → region label
ISO3_TO_REGION = {iso: reg for reg, isos in WHO_REGIONS.items() for iso in isos}

# Label encode region (consistent across both flu + dengue)
REGION_ENC = {'AFR': 0, 'AMR': 1, 'EMR': 2, 'EUR': 3, 'SEAR': 4, 'WPR': 5, 'UNK': 6}

def enrich_features(in_path, out_path):
    df = pd.read_csv(in_path)
    # who_region_enc
    df['who_region']     = df['iso3'].map(ISO3_TO_REGION).fillna('UNK')
    df['who_region_enc'] = df['who_region'].map(REGION_ENC).astype(int)
    # quarter: q1=week 1-13, q2=14-26, q3=27-39, q4=40-52
    df['quarter']        = ((df['iso_week'] - 1) // 13 + 1).clip(1, 4)
    n_unk = (df['who_region'] == 'UNK').sum()
    countries_unk = df.loc[df['who_region'] == 'UNK', 'iso3'].unique().tolist()
    df = df.drop(columns=['who_region'])
    df.to_csv(out_path, index=False)
    print(f'  Saved → {out_path.name}')
    print(f'  Shape: {df.shape}')
    print(f'  Region distribution: {df["who_region_enc"].value_counts().sort_index().to_dict()}')
    if n_unk > 0:
        print(f'  ⚠️ {n_unk} rows có iso3 chưa map ({len(countries_unk)} countries): {countries_unk[:20]}')
    return df

print('=== FLU ===')
flu_df_enriched = enrich_features(PROCESSED / 'features_flu_v3.csv', PROCESSED / 'features_flu_v3.csv')

print('\n=== DENGUE ===')
dengue_df_enriched = enrich_features(PROCESSED / 'features_dengue_v3.csv', PROCESSED / 'features_dengue_v3.csv')

print('\n✅ Đã thêm who_region_enc + quarter vào features_v3.')
print('   Chạy lại SESSION 9 (label gen) + SESSION 10 (training) để dùng features mới.')

📌 **[8.4]** Thêm 2 features đã proven work từ approach cũ (roadmap item #5): 
(1) **`who_region_enc`** = WHO region encoded (AFR=0, AMR=1, EMR=2, EUR=3, SEAR=4, WPR=5) — group countries theo vùng dịch tễ. (2) **`quarter`** = quý của năm (1-4 từ iso_week) — encode mùa thô bên cạnh week_sin/cos. Mapping ISO3→region từ WHO official list (194 countries). UNK = country không trong list (cảnh báo nếu có).

In [ ]:
# [8.5] log1p transform AR lag features (fix extreme skewness)
# inf_cases_lag*, dengue_total_lag* skewed nghiêm trọng (max/median ratio > 2000-13000)
# log1p giúp:
#   - LogReg: compress range, gradient ổn định hơn
#   - Tree-based: ít depth hơn, generalize tốt hơn
#   - Không thay đổi rank → không phá AR signal

AR_FLU    = ['inf_cases_lag1w', 'inf_cases_lag2w', 'inf_cases_lag4w']
AR_DENGUE = ['dengue_total_lag1w', 'dengue_total_lag2w', 'dengue_total_lag4w']

def log1p_ar(in_path, ar_cols):
    df = pd.read_csv(in_path)
    print(f'  Before log1p:')
    for c in ar_cols:
        print(f'    {c:25s} med={df[c].median():8.2f} max={df[c].max():12.2f}')
    for c in ar_cols:
        df[c] = np.log1p(df[c])
    print(f'  After log1p:')
    for c in ar_cols:
        print(f'    {c:25s} med={df[c].median():8.2f} max={df[c].max():12.2f}')
    df.to_csv(in_path, index=False)
    print(f'  Saved → {in_path.name}')

print('=== FLU ===')
log1p_ar(PROCESSED / 'features_flu_v3.csv', AR_FLU)

print('\n=== DENGUE ===')
log1p_ar(PROCESSED / 'features_dengue_v3.csv', AR_DENGUE)

print('\n✅ Đã log1p AR features. Re-run [9.0]→[9.3]→[10.1]→[10.8] với features đã transform.')

📌 **[8.5]** Fix skewness AR lag features bằng `log1p` (Brazil dengue 146,906 ca → 11.9 sau log1p). Lý do: AR lag max/median ratio > 2,000–13,000 — vượt xa ngưỡng outlier. log1p preserve rank (tree split không đổi) nhưng compress range (LogReg gradient ổn định). Note: KHÔNG transform target/label — đó là ordinal 3-class.

---

# 🏷️ SESSION 9 — ENDEMIC CHANNEL LABEL GENERATION
> **Input:** `features_flu_v3.csv`, `features_dengue_v3.csv`
> **Output:** `features_flu_labeled.csv`, `features_dengue_labeled.csv`
>
> Tạo nhãn Low/Medium/High theo phương pháp Endemic Channel (Bortman 1999 / WHO EWARS).
> Per (iso3, iso_week): baseline = mean(5 năm cùng tuần), upper = baseline + 2σ.

In [ ]:
# [9.0] RESTART — load feature files
flu_df    = pd.read_csv(PROCESSED / 'features_flu_v3.csv')
dengue_df = pd.read_csv(PROCESSED / 'features_dengue_v3.csv')

print(f'Flu features:    {flu_df.shape}')
print(f'Dengue features: {dengue_df.shape}')
print(f'Flu years:    {sorted(flu_df["iso_year"].unique())}')
print(f'Dengue years: {sorted(dengue_df["iso_year"].unique())}')


📌 **[9.0]** Load 2 feature files từ SESSION 8. Kiểm tra shape và năm để đảm bảo đọc đúng file.

In [ ]:
# [9.1] Generate Endemic Channel labels — tất cả nước, tất cả tuần

def generate_endemic_labels(df, target_col, min_hist=5):
    """
    Bortman 1999 / WHO EWARS:
      baseline = mean(cases, cùng iso_week, 5 năm trước)
      upper    = baseline + 2 * std
      Low    : cases < baseline
      Medium : baseline <= cases < upper
      High   : cases >= upper
    Edge case: nếu baseline=sd=0 (all-zero history) → cases=0 → Low, cases>0 → High
    """
    df = df.sort_values(['iso3','iso_year','iso_week']).copy()
    df['risk_label']   = None
    df['ec_baseline']  = np.nan
    df['ec_upper']     = np.nan

    for iso3, grp in df.groupby('iso3'):
        for idx, row in grp.iterrows():
            yr, wk = row['iso_year'], row['iso_week']
            hist = grp[
                (grp['iso_week'] == wk) &
                (grp['iso_year'] < yr) &
                (grp['iso_year'] >= yr - 5)
            ][target_col]

            if len(hist) < min_hist:
                continue

            baseline = hist.mean()
            sd       = hist.std()
            upper    = baseline + 2 * sd
            cases    = row[target_col] if not pd.isna(row[target_col]) else 0

            df.loc[idx, 'ec_baseline'] = round(baseline, 2)
            df.loc[idx, 'ec_upper']    = round(upper, 2)

            # Edge case: all-zero history → baseline=upper=0
            if baseline == 0 and sd == 0:
                df.loc[idx, 'risk_label'] = 'Low' if cases == 0 else 'High'
            else:
                df.loc[idx, 'risk_label'] = (
                    'High'   if cases >= upper    else
                    'Medium' if cases >= baseline else
                    'Low'
                )

    return df

print('Generating flu labels...')
flu_labeled = generate_endemic_labels(flu_df, 'inf_cases')
flu_lab = flu_labeled.dropna(subset=['risk_label'])
print(f'  Labeled: {len(flu_lab):,} / {len(flu_df):,} rows')
print(f'  Distribution: {flu_lab["risk_label"].value_counts().to_dict()}')
pct = flu_lab["risk_label"].value_counts(normalize=True)*100
print(f'  %: {pct.round(1).to_dict()}')

print('\nGenerating dengue labels...')
dengue_labeled = generate_endemic_labels(dengue_df, 'dengue_total')
dengue_lab = dengue_labeled.dropna(subset=['risk_label'])
print(f'  Labeled: {len(dengue_lab):,} / {len(dengue_df):,} rows')
print(f'  Distribution: {dengue_lab["risk_label"].value_counts().to_dict()}')
pct_d = dengue_lab["risk_label"].value_counts(normalize=True)*100
print(f'  %: {pct_d.round(1).to_dict()}')


📌 **[9.1]** Generate Endemic Channel labels cho toàn bộ dataset.

- **Flu:** 20,552 rows labeled — Low 57% / Medium 26% / High 17%
- **Dengue:** 21,739 rows labeled — Low 87% / Medium 6% / High 7%

35.6% rows không được label vì chưa đủ 5 năm lịch sử (2010–2014 và nước báo không liên tục) — đúng thiết kế, không phải lỗi. Fix edge case all-zero history: nước không có dengue 5 năm + tuần này cũng 0 → Low (không phải High như lần chạy đầu).

In [ ]:
# [9.2] Kiểm tra class balance + save labeled CSV

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, df_lab) in zip(axes, [('Flu', flu_lab), ('Dengue', dengue_lab)]):
    counts = df_lab['risk_label'].value_counts()[['Low','Medium','High']]
    pcts   = counts / counts.sum() * 100
    bars = ax.bar(['Low','Medium','High'], pcts.values,
                  color=['#4CAF50','#FF9800','#F44336'])
    for bar, pct in zip(bars, pcts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', fontsize=10)
    ax.set_title(f'{name} — Label distribution')
    ax.set_ylabel('%')
    ax.set_ylim(0, 100)

plt.suptitle('SESSION 9 — Endemic Channel Label Balance', fontsize=12)
plt.tight_layout()
plt.show()

# Save
FLU_LABELED    = PROCESSED / 'features_flu_labeled.csv'
DENGUE_LABELED = PROCESSED / 'features_dengue_labeled.csv'
flu_lab.to_csv(FLU_LABELED, index=False)
dengue_lab.to_csv(DENGUE_LABELED, index=False)
print(f'Saved: {FLU_LABELED.name} — {len(flu_lab):,} rows')
print(f'Saved: {DENGUE_LABELED.name} — {len(dengue_lab):,} rows')


📌 **[9.2]** Visualize class balance và save file labeled. Kỳ vọng: Low chiếm đa số (~50%), High ít nhất (~10–15%). Nếu balance quá lệch → cần `class_weight='balanced'` khi train (đã quyết định từ trước). Output: 2 file CSV có thêm cột `risk_label`, `ec_baseline`, `ec_upper` — input trực tiếp cho SESSION 10 (training).

In [ ]:
# [9.3] Filter year >= 2015 + dengue endemic countries + save labeled CSV

# Filter 1: year >= 2015 (2010-2014 = warm-up, không có label hợp lệ)
flu_lab    = flu_lab[flu_lab['iso_year'] >= 2015].copy()
dengue_lab = dengue_lab[dengue_lab['iso_year'] >= 2015].copy()

# Filter 2: dengue chỉ giữ nước endemic (>= 1 tuần dengue_total > 0)
valid_dengue_iso3 = (
    dengue_lab.groupby('iso3')['dengue_total']
              .max()
              .loc[lambda x: x > 0]
              .index
)
n_before = dengue_lab['iso3'].nunique()
dengue_lab = dengue_lab[dengue_lab['iso3'].isin(valid_dengue_iso3)].copy()
print(f'Dengue countries: {n_before} -> {dengue_lab["iso3"].nunique()} (drop non-endemic)')

print('\n=== FLU ===')
print(f"Rows: {len(flu_lab):,} | Countries: {flu_lab['iso3'].nunique()} | Years: {sorted(flu_lab['iso_year'].unique())}")
print(flu_lab['risk_label'].value_counts(normalize=True).round(3))

print('\n=== DENGUE ===')
print(f"Rows: {len(dengue_lab):,} | Countries: {dengue_lab['iso3'].nunique()} | Years: {sorted(dengue_lab['iso_year'].unique())}")
print(dengue_lab['risk_label'].value_counts(normalize=True).round(3))

# Save (overwrite)
FLU_LABELED    = PROCESSED / 'features_flu_labeled.csv'
DENGUE_LABELED = PROCESSED / 'features_dengue_labeled.csv'
flu_lab.to_csv(FLU_LABELED, index=False)
dengue_lab.to_csv(DENGUE_LABELED, index=False)
print(f'\nSaved: {FLU_LABELED.name}')
print(f'Saved: {DENGUE_LABELED.name}')

---

# 🤖 SESSION 10 — XGBClassifier Training (Walk-forward CV)
> **Input:** `features_flu_labeled.csv`, `features_dengue_labeled.csv`
> **Output:** `xgb_flu_clf.pkl`, `xgb_dengue_clf.pkl`, CV metrics
>
> Train ordinal multi-class classifier (Low/Medium/High) với walk-forward CV.
> Reference: Wellcome Open Research 2024 (ordinal influenza classification).

In [ ]:
# [10.1] Feature check + NaN scan (sau khi co log1p + who_region_enc + quarter)
# PROCESSED duoc define o [0.4]: BASE / 'dataset' / 'processed'

import pandas as pd
import numpy as np

flu_lab    = pd.read_csv(PROCESSED / 'features_flu_labeled.csv')
dengue_lab = pd.read_csv(PROCESSED / 'features_dengue_labeled.csv')

NON_FEATURES = {
    'iso3','iso_year','iso_week',
    'risk_label','ec_baseline','ec_upper','ec_sd',
    'inf_cases','inf_total','INF_A','INF_B','flu_total','dengue_total'
}

flu_features    = [c for c in flu_lab.columns    if c not in NON_FEATURES]
dengue_features = [c for c in dengue_lab.columns if c not in NON_FEATURES]

print(f"FLU features ({len(flu_features)}):    {flu_features}")
print(f"DENGUE features ({len(dengue_features)}): {dengue_features}")
print()

# NaN check
flu_nan = flu_lab[flu_features].isna().sum()
dengue_nan = dengue_lab[dengue_features].isna().sum()
print("FLU NaN:", flu_nan[flu_nan > 0].to_dict() if flu_nan.any() else "None")
print("DENGUE NaN:", dengue_nan[dengue_nan > 0].to_dict() if dengue_nan.any() else "None")
print()

# Verify log1p da vao labeled CSV (max phai ~9-12, khong phai 13000+)
print("FLU AR max (ky vong ~9-10 sau log1p):")
for c in ['inf_cases_lag1w','inf_cases_lag2w','inf_cases_lag4w']:
    if c in flu_lab.columns:
        print(f"  {c}: max={flu_lab[c].max():.2f}  median={flu_lab[c].median():.2f}")

print("DENGUE AR max (ky vong ~11-12 sau log1p):")
for c in ['dengue_total_lag1w','dengue_total_lag2w','dengue_total_lag4w']:
    if c in dengue_lab.columns:
        print(f"  {c}: max={dengue_lab[c].max():.2f}  median={dengue_lab[c].median():.2f}")
print()
print("who_region_enc distribution (flu):")
print(flu_lab['who_region_enc'].value_counts().sort_index().to_dict())

📌 **[10.1]** Liệt kê feature columns và kiểm tra NaN trước khi train. Loại bỏ metadata (`iso3`, `iso_year`, `iso_week`), target (`risk_label`), raw cases (`flu_total`, `dengue_total`), và endemic baseline metadata (`ec_baseline`, `ec_upper`). Nếu có NaN → cần investigate trước khi train, không được fillna tùy tiện.

In [ ]:
# [10.2] Setup walk-forward CV + helper functions

import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

# --- Label encoding (ordinal: Low=0 < Medium=1 < High=2) ---
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
LABEL_INV = {v: k for k, v in LABEL_MAP.items()}

flu_lab['label']    = flu_lab['risk_label'].map(LABEL_MAP)
dengue_lab['label'] = dengue_lab['risk_label'].map(LABEL_MAP)

# Drop NaN labels (rows thiếu history → baseline=NaN → label=NaN)
n_flu_before, n_den_before = len(flu_lab), len(dengue_lab)
flu_lab    = flu_lab.dropna(subset=['label']).reset_index(drop=True)
dengue_lab = dengue_lab.dropna(subset=['label']).reset_index(drop=True)
flu_lab['label']    = flu_lab['label'].astype(int)
dengue_lab['label'] = dengue_lab['label'].astype(int)
print(f'Flu:    {n_flu_before:,} → {len(flu_lab):,} (dropped {n_flu_before-len(flu_lab):,} NaN labels)')
print(f'Dengue: {n_den_before:,} → {len(dengue_lab):,} (dropped {n_den_before-len(dengue_lab):,} NaN labels)')

# --- Walk-forward CV scheme ---
VAL_YEARS = [2017, 2018, 2019]

def walk_forward_splits(df, val_years=VAL_YEARS):
    '''Yield (val_year, train_idx, val_idx). Train trên tất cả năm < val_year.'''
    for val_year in val_years:
        train_idx = df.index[df['iso_year'] < val_year]
        val_idx   = df.index[df['iso_year'] == val_year]
        yield val_year, train_idx, val_idx

# --- Metric computation ---
def compute_metrics(y_true, y_pred, y_proba=None):
    m = {
        'macro_f1':    f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    f1_per = f1_score(y_true, y_pred, average=None, labels=[0,1,2], zero_division=0)
    m['f1_low'], m['f1_medium'], m['f1_high'] = f1_per
    if y_proba is not None and len(np.unique(y_true)) == 3:
        try:
            m['auc_ovr'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro', labels=[0,1,2])
        except Exception:
            m['auc_ovr'] = np.nan
    else:
        m['auc_ovr'] = np.nan
    return m

# --- Generic walk-forward runner ---
def run_cv(df, features, model_factory, model_name, use_sample_weight=True, verbose=True):
    '''
    model_factory: callable trả về sklearn-compatible estimator mới (fit/predict/predict_proba).
    use_sample_weight: True → áp dụng class_weight='balanced' qua sample_weight.
    Returns: list of dict, mỗi dict là metrics của 1 fold.
    '''
    folds = []
    for val_year, tr_idx, va_idx in walk_forward_splits(df):
        X_tr = df.loc[tr_idx, features].values
        y_tr = df.loc[tr_idx, 'label'].values
        X_va = df.loc[va_idx, features].values
        y_va = df.loc[va_idx, 'label'].values

        model = model_factory()
        if use_sample_weight:
            sw = compute_sample_weight('balanced', y_tr)
            try:
                model.fit(X_tr, y_tr, sample_weight=sw)
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            model.fit(X_tr, y_tr)

        y_pred  = model.predict(X_va)
        y_proba = model.predict_proba(X_va) if hasattr(model, 'predict_proba') else None
        m = compute_metrics(y_va, y_pred, y_proba)
        m.update({'model': model_name, 'val_year': val_year,
                  'n_train': len(tr_idx), 'n_val': len(va_idx)})
        folds.append(m)
        if verbose:
            print(f"  {model_name:20s} val={val_year} | n_tr={len(tr_idx):6,} n_va={len(va_idx):5,} | "
                  f"macroF1={m['macro_f1']:.3f} | F1(L,M,H)=({m['f1_low']:.2f},{m['f1_medium']:.2f},{m['f1_high']:.2f}) | "
                  f"AUC={m['auc_ovr']:.3f}")
    return folds

# --- Sanity check year coverage ---
for disease, df in [('Flu', flu_lab), ('Dengue', dengue_lab)]:
    print(f'\n{disease}:')
    for vy, tr, va in walk_forward_splits(df):
        print(f'  val={vy}: train={len(tr):,} rows ({sorted(df.loc[tr,"iso_year"].unique())}) | val={len(va):,} rows')

print('\n✅ Setup xong. Sẵn sàng train từ [10.3].')

📌 **[10.2]** Setup foundation cho tất cả model: 
(1) Label encoding ordinal (Low=0 < Medium=1 < High=2 — giữ thứ tự để model nhận thấy thứ bậc nếu cần). (2) Walk-forward CV: train trên năm < val_year, validate năm val_year ∈ {2017, 2018, 2019} → 3 folds. (3) `compute_metrics`: macro-F1 (chính), per-class F1, AUC OvR — phù hợp imbalanced multi-class. (4) `run_cv`: generic runner, mọi model sau chỉ cần truyền `model_factory` lambda là chạy được — đảm bảo **cùng CV + cùng metrics** cho fair comparison.

In [ ]:
# [10.2b] Sanity checks trước khi train — phát hiện lỗi sớm

def sanity_checks(df, features, disease_name):
    print(f'\n{"="*70}')
    print(f'SANITY CHECKS — {disease_name}')
    print("="*70)
    issues = []

    # 1. Class balance overall + per fold
    print('\n[1] Class balance per fold:')
    for val_year, tr_idx, va_idx in walk_forward_splits(df):
        tr_dist = df.loc[tr_idx, 'label'].value_counts(normalize=True).sort_index()
        va_dist = df.loc[va_idx, 'label'].value_counts(normalize=True).sort_index()
        tr_str = ', '.join([f'{LABEL_INV[k]}={v:.2f}' for k,v in tr_dist.items()])
        va_str = ', '.join([f'{LABEL_INV[k]}={v:.2f}' for k,v in va_dist.items()])
        print(f'  val={val_year} | train: {tr_str} | val: {va_str}')
        # Warning: fold thiếu class
        if len(va_dist) < 3:
            issues.append(f'⚠️ Val {val_year}: chỉ có {len(va_dist)} class')
        if len(tr_dist) < 3:
            issues.append(f'⚠️ Train {val_year}: chỉ có {len(tr_dist)} class')

    # 2. Country overlap train/val (val countries phải có trong train cho AR lag)
    print('\n[2] Country overlap train/val:')
    for val_year, tr_idx, va_idx in walk_forward_splits(df):
        tr_iso = set(df.loc[tr_idx, 'iso3'].unique())
        va_iso = set(df.loc[va_idx, 'iso3'].unique())
        new_in_val = va_iso - tr_iso
        coverage  = len(va_iso & tr_iso) / len(va_iso) * 100
        print(f'  val={val_year} | train={len(tr_iso)} nước, val={len(va_iso)} nước, coverage={coverage:.0f}%')
        if new_in_val:
            issues.append(f'⚠️ Val {val_year} có {len(new_in_val)} nước MỚI không có trong train: {sorted(new_in_val)[:5]}')

    # 3. Feature scale check
    print('\n[3] Feature scale (min/median/max):')
    for f in features:
        v = df[f]
        ratio = abs(v.max()) / (abs(v.median()) + 1e-9)
        flag  = ' ⚠️ extreme range' if ratio > 1e4 else ''
        print(f'  {f:30s} min={v.min():10.2f} med={v.median():10.2f} max={v.max():12.2f}{flag}')
        if ratio > 1e4:
            issues.append(f'⚠️ Feature {f} có range cực đoan — consider log/clip')

    # 4. No index overlap (data leakage check)
    print('\n[4] Train/val index overlap (data leakage):')
    for val_year, tr_idx, va_idx in walk_forward_splits(df):
        overlap = set(tr_idx) & set(va_idx)
        print(f'  val={val_year} | overlap rows: {len(overlap)} ' + ('✅' if not overlap else '🚨 LEAKAGE'))
        if overlap:
            issues.append(f'🚨 LEAKAGE val {val_year}: {len(overlap)} rows trong cả train + val')

    # 5. Summary
    print('\n[5] Summary:')
    if not issues:
        print('  ✅ Tất cả checks pass — sẵn sàng train')
    else:
        print(f'  ⚠️ {len(issues)} issue(s):')
        for i in issues:
            print(f'    - {i}')
    return issues

flu_issues    = sanity_checks(flu_lab,    flu_features,    'FLU')
dengue_issues = sanity_checks(dengue_lab, dengue_features, 'DENGUE')

📌 **[10.2b]** Sanity checks — phát hiện sớm 4 loại lỗi: 
(1) **Class balance per fold** — fold thiếu class sẽ làm F1 sai số; (2) **Country overlap** — val countries phải có trong train cho AR lag work; (3) **Feature scale extreme** — flag features cần log/clip (bảo vệ LogReg); (4) **Data leakage** — train/val index overlap. Nếu pass tất cả → confident train. Nếu warning → fix trước khi tốn compute Optuna.

In [ ]:
# [10.3] Naive baselines — Majority class + Same-week-last-year

from sklearn.dummy import DummyClassifier

# === 1. MAJORITY CLASS BASELINE ===
print('=' * 70)
print('MAJORITY CLASS BASELINE (always predict most frequent class)')
print('=' * 70)

print('\n--- FLU ---')
flu_majority = run_cv(flu_lab, flu_features,
                       lambda: DummyClassifier(strategy='most_frequent'),
                       'Majority',
                       use_sample_weight=False)

print('\n--- DENGUE ---')
dengue_majority = run_cv(dengue_lab, dengue_features,
                          lambda: DummyClassifier(strategy='most_frequent'),
                          'Majority',
                          use_sample_weight=False)

# === 2. SAME-WEEK-LAST-YEAR BASELINE ===
def run_swly(df, val_years=VAL_YEARS):
    '''Predict label[t] = label of same (iso3, iso_week) in previous year.
       Fallback: train-set majority class if no match.'''
    folds = []
    for val_year in val_years:
        train_df = df[df['iso_year'] <  val_year]
        val_df   = df[df['iso_year'] == val_year]
        prev_df  = df[df['iso_year'] == val_year - 1]
        lookup = prev_df.set_index(['iso3', 'iso_week'])['label'].to_dict()
        majority = int(train_df['label'].mode().iloc[0])
        keys = list(zip(val_df['iso3'], val_df['iso_week']))
        y_pred = np.array([lookup.get(k, majority) for k in keys])
        y_va   = val_df['label'].values
        m = compute_metrics(y_va, y_pred, y_proba=None)
        m.update({'model': 'SameWeekLastYear', 'val_year': val_year,
                  'n_train': len(train_df), 'n_val': len(val_df)})
        folds.append(m)
        print(f"  SameWeekLastYear     val={val_year} | n_va={len(val_df):5,} | "
              f"macroF1={m['macro_f1']:.3f} | "
              f"F1(L,M,H)=({m['f1_low']:.2f},{m['f1_medium']:.2f},{m['f1_high']:.2f})")
    return folds

print('\n' + '=' * 70)
print('SAME-WEEK-LAST-YEAR BASELINE')
print('=' * 70)

print('\n--- FLU ---')
flu_swly = run_swly(flu_lab)

print('\n--- DENGUE ---')
dengue_swly = run_swly(dengue_lab)

📌 **[10.3]** 2 naive baselines — đặt **sàn** để so sánh các model thực: (1) **Majority** = luôn predict class phổ biến nhất (Low) → macroF1 thấp (~0.22) vì F1(Medium)=F1(High)=0. (2) **Same-week-last-year** = giả định pattern lặp theo năm. Tận dụng tính chu kỳ của dịch bệnh. Nếu model phức tạp không đánh bại được SWLY → có vấn đề (overfitting hoặc feature kém).

In [ ]:
# [10.4] Logistic Regression (statistical baseline)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def make_logreg():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            multi_class='multinomial', solver='lbfgs',
            class_weight='balanced',
            max_iter=2000, C=1.0, random_state=42))
    ])

print('=' * 70)
print('LOGISTIC REGRESSION (multinomial, class_weight=balanced, StandardScaler)')
print('=' * 70)

# use_sample_weight=False vì Pipeline không nhận trực tiếp — đã set class_weight='balanced' trong estimator
print('\n--- FLU ---')
flu_logreg = run_cv(flu_lab, flu_features, make_logreg, 'LogReg', use_sample_weight=False)

print('\n--- DENGUE ---')
dengue_logreg = run_cv(dengue_lab, dengue_features, make_logreg, 'LogReg', use_sample_weight=False)

📌 **[10.4]** Logistic Regression — **statistical baseline** chuẩn cho multi-class classification. Dùng `StandardScaler` vì LogReg sensitive với scale (AR cases lên đến hàng nghìn, week_sin chỉ [-1,1]). `multinomial` = optimize cross-entropy đúng cho 3 class (khác `ovr` rời rạc). Sample weight 'balanced' xử lý imbalance. **Kỳ vọng:** macroF1 ~ 0.45–0.55 (vượt SWLY), nhưng kém XGBoost vì model tuyến tính không bắt được interaction giữa weather × cases.

In [ ]:
# [10.5] Random Forest (tree-based baseline)

from sklearn.ensemble import RandomForestClassifier

def make_rf():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
    )

print('=' * 70)
print('RANDOM FOREST (n=300, depth=12, balanced)')
print('=' * 70)

print('\n--- FLU ---')
flu_rf = run_cv(flu_lab, flu_features, make_rf, 'RandomForest', use_sample_weight=False)

print('\n--- DENGUE ---')
dengue_rf = run_cv(dengue_lab, dengue_features, make_rf, 'RandomForest', use_sample_weight=False)

📌 **[10.5]** Random Forest — **tree-based baseline**. Bắt được interaction phi tuyến (weather × cases) mà LogReg miss. `max_depth=12` tránh overfit (data ~7-16k rows train); `min_samples_leaf=5` regularize. `class_weight='balanced'` set trực tiếp trong estimator. **Kỳ vọng:** macroF1 ~ 0.55–0.65 (vượt LogReg đáng kể).

In [ ]:
# [10.6] XGBClassifier (boosted tree — primary candidate)

from xgboost import XGBClassifier

def make_xgb():
    return XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric='mlogloss',
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        verbosity=0,
    )

print('=' * 70)
print('XGBOOST (multi:softprob, n=500, depth=6, lr=0.05, balanced)')
print('=' * 70)

print('\n--- FLU ---')
flu_xgb = run_cv(flu_lab, flu_features, make_xgb, 'XGBoost', use_sample_weight=True)

print('\n--- DENGUE ---')
dengue_xgb = run_cv(dengue_lab, dengue_features, make_xgb, 'XGBoost', use_sample_weight=True)

📌 **[10.6]** XGBClassifier — **model chính dự kiến production**. `multi:softprob` = output P(Low), P(Med), P(High) trực tiếp (khớp yêu cầu 'khả năng diễn ra' của khoa). `max_depth=6` + `lr=0.05` + `n_est=500` là sweet spot cho dataset ~7-16k rows (tránh overfit). `subsample=0.8` + `colsample=0.8` = bagging-like regularize. Sample weight 'balanced' qua `run_cv`. **Kỳ vọng:** macroF1 ~ 0.55–0.70, AUC ~ 0.80–0.90 — vượt RF.

In [ ]:
# [10.7] LightGBM (boosted tree — alternative)

from lightgbm import LGBMClassifier

def make_lgbm():
    return LGBMClassifier(
        objective='multiclass',
        num_class=3,
        n_estimators=500,
        max_depth=6,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
        verbosity=-1,
    )

print('=' * 70)
print('LIGHTGBM (multiclass, n=500, depth=6, lr=0.05, balanced)')
print('=' * 70)

# class_weight đã set trong estimator → use_sample_weight=False
print('\n--- FLU ---')
flu_lgbm = run_cv(flu_lab, flu_features, make_lgbm, 'LightGBM', use_sample_weight=False)

print('\n--- DENGUE ---')
dengue_lgbm = run_cv(dengue_lab, dengue_features, make_lgbm, 'LightGBM', use_sample_weight=False)

📌 **[10.7]** LightGBM — boosted tree khác XGBoost. Histogram-based + leaf-wise (XGBoost = level-wise) → train nhanh hơn 2-3×. `num_leaves=31` (default) là sweet spot. Hyperparams cố tình giống XGBoost để **fair comparison** kiến trúc khác nhau. **Kỳ vọng:** macroF1 tương đương XGBoost (~0.48–0.55) — leaf-wise có thể bắt được pattern khác.

In [ ]:
# [10.8] Comparison table — aggregate metrics across folds & save

import pandas as pd
import joblib
import time

# Combine all fold-level metrics into one DataFrame
all_results = {
    'Flu': {
        'Majority':         flu_majority,
        'SameWeekLastYear': flu_swly,
        'LogReg':           flu_logreg,
        'RandomForest':     flu_rf,
        'XGBoost':          flu_xgb,
        'LightGBM':         flu_lgbm,
    },
    'Dengue': {
        'Majority':         dengue_majority,
        'SameWeekLastYear': dengue_swly,
        'LogReg':           dengue_logreg,
        'RandomForest':     dengue_rf,
        'XGBoost':          dengue_xgb,
        'LightGBM':         dengue_lgbm,
    },
}

rows = []
for disease, models in all_results.items():
    for model_name, folds in models.items():
        df_f = pd.DataFrame(folds)
        rows.append({
            'disease':      disease,
            'model':        model_name,
            'macro_f1':     df_f['macro_f1'].mean(),
            'macro_f1_std': df_f['macro_f1'].std(),
            'f1_low':       df_f['f1_low'].mean(),
            'f1_medium':    df_f['f1_medium'].mean(),
            'f1_high':      df_f['f1_high'].mean(),
            'auc_ovr':      df_f.get('auc_ovr', pd.Series([float('nan')])).mean(),
        })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values(['disease', 'macro_f1'], ascending=[True, False]).reset_index(drop=True)

print('=' * 90)
print('SESSION 10 — MODEL COMPARISON (mean across 3 walk-forward folds)')
print('=' * 90)
for disease in ['Flu', 'Dengue']:
    print(f'\n--- {disease.upper()} ---')
    sub = results_df[results_df['disease'] == disease].drop(columns=['disease'])
    print(sub.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# Save metrics
METRICS_PATH = PROCESSED / 'session10_metrics.csv'
results_df.to_csv(METRICS_PATH, index=False)
print(f'\nSaved metrics: {METRICS_PATH.name}')

# --- Champion selection: best macroF1 per disease ---
champions = {}
for disease in ['Flu', 'Dengue']:
    sub = results_df[(results_df['disease'] == disease) & 
                     (~results_df['model'].isin(['Majority', 'SameWeekLastYear']))]
    champ = sub.iloc[0]
    champions[disease] = champ['model']
    print(f"\n🏆 {disease} champion: {champ['model']} (macroF1={champ['macro_f1']:.3f}, AUC={champ['auc_ovr']:.3f})")

# --- Re-train champion on FULL labeled data (2015-2019) & save model ---
MODEL_FACTORIES = {
    'LogReg':       make_logreg,
    'RandomForest': make_rf,
    'XGBoost':      make_xgb,
    'LightGBM':     make_lgbm,
}

for disease, df_lab, features in [
    ('Flu',    flu_lab,    flu_features),
    ('Dengue', dengue_lab, dengue_features),
]:
    champ_name = champions[disease]
    model = MODEL_FACTORIES[champ_name]()
    X = df_lab[features].values
    y = df_lab['label'].values
    from sklearn.utils.class_weight import compute_sample_weight
    sw = compute_sample_weight('balanced', y)
    t0 = time.time()
    try:
        model.fit(X, y, sample_weight=sw)
    except (TypeError, ValueError):
        model.fit(X, y)
    train_time = time.time() - t0
    out_path = PROCESSED / f'champion_{disease.lower()}_{champ_name.lower()}.pkl'
    joblib.dump({'model': model, 'features': features, 'label_map': LABEL_MAP},
                out_path)
    print(f'Saved {disease} champion ({champ_name}, train_time={train_time:.1f}s) → {out_path.name}')

📌 **[10.8]** Tổng kết SESSION 10: 
(1) **Bảng so sánh 6 model × 2 disease × CV folds** — mean macroF1, per-class F1, AUC OvR. (2) **Champion selection**: macroF1 cao nhất (loại 2 naive baseline khỏi candidate). (3) **Re-train champion trên full 2015–2019** (không CV) để model production có max data. (4) **Save** model + features + label_map qua joblib → ready cho SESSION 11 (evaluation 2022) và SESSION 12 (deploy).

---

# 🎯 SESSION 11 — Hyperparameter Tuning (Optuna)
> Tune XGBoost + LightGBM bằng Bayesian optimization (Optuna TPE sampler).
> Objective: maximize mean macro_f1 across 3 walk-forward folds.
> Reference: Akiba et al. 2019 (Optuna paper).

In [ ]:
# [11.1] Optuna tuning XGBoost (flu + dengue) + retrain champion

import optuna
import time
import json as _json
from xgboost import XGBClassifier
import pickle
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30

def objective_xgb(trial, df, features):
    params = {
        'objective':        'multi:softprob',
        'num_class':        3,
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800, step=100),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'eval_metric':      'mlogloss',
        'tree_method':      'hist',
        'n_jobs':           -1,
        'random_state':     42,
        'verbosity':        0,
    }
    factory = lambda: XGBClassifier(**params)
    folds = run_cv(df, features, factory, 'XGB_trial', use_sample_weight=True, verbose=False)
    return np.mean([f['macro_f1'] for f in folds])

# --- FLU ---
print(f'Tuning XGBoost FLU --- {N_TRIALS} trials...')
t0 = time.time()
study_xgb_flu = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb_flu.optimize(lambda t: objective_xgb(t, flu_lab, flu_features),
                       n_trials=N_TRIALS, show_progress_bar=True)
flu_time = (time.time() - t0) / 60
print(f'Flu best macroF1 (Optuna): {study_xgb_flu.best_value:.4f}')
print(f'Best params: {study_xgb_flu.best_params}')
print(f'Time: {flu_time:.1f} min')

# --- DENGUE ---
print(f'
Tuning XGBoost DENGUE --- {N_TRIALS} trials...')
t0 = time.time()
study_xgb_dengue = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb_dengue.optimize(lambda t: objective_xgb(t, dengue_lab, dengue_features),
                           n_trials=N_TRIALS, show_progress_bar=True)
dengue_time = (time.time() - t0) / 60
print(f'Dengue best macroF1 (Optuna): {study_xgb_dengue.best_value:.4f}')
print(f'Best params: {study_xgb_dengue.best_params}')
print(f'Time: {dengue_time:.1f} min')

# --- Save best params ---
best_params_path = PROCESSED / 'xgb_best_params.json'
with open(best_params_path, 'w') as fp:
    _json.dump({'flu': study_xgb_flu.best_params, 'dengue': study_xgb_dengue.best_params}, fp, indent=2)
print(f'
Saved best params -> {best_params_path.name}')

# --- Retrain on full data + save champion if better than current ---
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
BASELINE_FLU_F1   = 0.485  # RF champion (session 10)
BASELINE_DENGUE_F1 = 0.496  # XGB untuned champion (session 10)

def retrain_xgb(df, features, best_params, disease):
    params = dict(best_params)
    params.update({'objective':'multi:softprob','num_class':3,
                   'eval_metric':'mlogloss','tree_method':'hist',
                   'n_jobs':-1,'random_state':42,'verbosity':0})
    X = df[features]
    y = df['risk_label'].map(LABEL_MAP)
    clf = XGBClassifier(**params)
    clf.fit(X, y)
    return clf

print('
--- Retrain FLU XGB tuned on full data ---')
flu_xgb_tuned = retrain_xgb(flu_lab, flu_features, study_xgb_flu.best_params, 'flu')
flu_xgb_f1 = study_xgb_flu.best_value
print(f'  Tuned XGB Flu macroF1={flu_xgb_f1:.4f} vs RF baseline={BASELINE_FLU_F1:.4f}')
if flu_xgb_f1 > BASELINE_FLU_F1:
    pkl_path = PROCESSED / 'champion_flu_xgboost_tuned.pkl'
    with open(pkl_path, 'wb') as fp:
        pickle.dump({'model': flu_xgb_tuned, 'features': flu_features,
                     'label_map': LABEL_MAP, 'macro_f1': flu_xgb_f1}, fp)
    print(f'  XGB TUNED beats RF -> saved {pkl_path.name}')
else:
    print(f'  RF remains champion for flu')

print('
--- Retrain DENGUE XGB tuned on full data ---')
dengue_xgb_tuned = retrain_xgb(dengue_lab, dengue_features, study_xgb_dengue.best_params, 'dengue')
dengue_xgb_f1 = study_xgb_dengue.best_value
print(f'  Tuned XGB Dengue macroF1={dengue_xgb_f1:.4f} vs untuned baseline={BASELINE_DENGUE_F1:.4f}')
pkl_path = PROCESSED / 'champion_dengue_xgboost_tuned.pkl'
with open(pkl_path, 'wb') as fp:
    pickle.dump({'model': dengue_xgb_tuned, 'features': dengue_features,
                 'label_map': LABEL_MAP, 'macro_f1': dengue_xgb_f1}, fp)
print(f'  Saved {pkl_path.name}')

print('
=== SUMMARY ===')
print(f'Flu   : RF={BASELINE_FLU_F1:.4f} | XGB tuned={flu_xgb_f1:.4f}')
print(f'Dengue: XGB untuned={BASELINE_DENGUE_F1:.4f} | XGB tuned={dengue_xgb_f1:.4f}')

📌 **[11.1]** Optuna TPE sampler tune XGBoost. Search space 10 hyperparams; N_TRIALS=30 đủ explore. Objective = mean macroF1 across 3 folds → fair vs RF chưa tune. Best params save JSON cho reproducibility.

---

# 🚧 SESSION 8–12 — SẼ ĐƯỢC THÊM SAU KHI CHẠY XONG SESSION 6 (EDA MỞ RỘNG)

> Sau khi EDA mở rộng (SESSION 3 tiếp — 8 bước) confirm data OK,
> Claude sẽ gửi lần lượt từng cell cho:
> - **SESSION 7**: Feature Engineering (weather lag + AR lag + seasonal + region)
> - **SESSION 8**: Endemic Channel Label Generation (Bortman 1999 / WHO EWARS)
> - **SESSION 9**: Train multi-model (Naive / Logistic / RandomForest / XGBClassifier / LightGBM)
> - **SESSION 10**: So sánh model — bảng metrics + chọn winner
> - **SESSION 11**: Eval chi tiết winner + export artifacts
>
> Lưu ý: SESSION 4 (ERA5) và 5 (Merge) là idempotent — đã có file output sẵn, chạy chỉ để verify.
> SESSION 7 (CCF) đã chạy ở v2, kết quả lag tối ưu đã chốt trong CLAUDE.md.


---

# SESSION 12 — Evaluation on 2022 Test Set (Post-COVID Generalization)
> Test XGBoost champions (flu + dengue) tren nam 2022 — out-of-sample, post-COVID.
>
> **Setup:**
> - ERA5 2022 co san (`era5_weekly_2022_final.csv`)
> - Flu 2022: tu `VIW_FNT.csv` (WHO FluNet, co san den 2026)
> - Dengue 2022: tu `National_extract_V1_3.csv` (OpenDengue, weekly T_res)
> - Endemic labels cho 2022: dung ec_baseline/upper trung binh tu training 2015-2019 (COVID-free)
> - Bat dau tu tuan 9 (lag8w can ERA5 week 1+ cua 2022)

In [ ]:
# [12.1] Load + process flu 2022 case data
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

RAW = BASE / 'dataset' / 'epidemic' / 'raw'

print("=== Check raw data files ===")
for fname in ['VIW_FNT.csv', 'National_extract_V1_3.csv']:
    p = RAW / fname
    ok = p.exists()
    mb = p.stat().st_size // (1024*1024) if ok else 0
    print(f"  {'OK' if ok else 'MISSING'} {fname}: {mb} MB")

print()
flu_raw = pd.read_csv(RAW / 'VIW_FNT.csv',
                      usecols=['COUNTRY_CODE','ISO_YEAR','ISO_WEEK','INF_A','INF_B'])
flu_raw = flu_raw[flu_raw['ISO_YEAR'] == 2022].copy()
flu_raw[['INF_A','INF_B']] = flu_raw[['INF_A','INF_B']].fillna(0)

flu_2022 = (flu_raw
            .groupby(['COUNTRY_CODE','ISO_YEAR','ISO_WEEK'], as_index=False)
            .agg({'INF_A':'sum','INF_B':'sum'}))
flu_2022['inf_cases'] = flu_2022['INF_A'] + flu_2022['INF_B']
flu_2022 = flu_2022.rename(columns={
    'COUNTRY_CODE':'iso3','ISO_YEAR':'iso_year','ISO_WEEK':'iso_week'})
flu_2022 = flu_2022[['iso3','iso_year','iso_week','inf_cases']]

print(f"Flu 2022: {len(flu_2022):,} rows | {flu_2022['iso3'].nunique()} countries | "
      f"weeks {flu_2022['iso_week'].min()}-{flu_2022['iso_week'].max()}")
print(f"Total cases: {flu_2022['inf_cases'].sum():,.0f}")
print(f"Countries with any cases: {(flu_2022.groupby('iso3')['inf_cases'].max() > 0).sum()}")


In [ ]:
# [12.2] Flu 2022: ERA5 merge + weather/AR lags + log1p + region + endemic label

WHO_REGIONS = {
    'AFR': ['DZA','AGO','BEN','BWA','BFA','BDI','CPV','CMR','CAF','TCD','COM','COG','COD','CIV',
            'GNQ','ERI','SWZ','ETH','GAB','GMB','GHA','GIN','GNB','KEN','LSO','LBR','MDG','MWI',
            'MLI','MRT','MUS','MOZ','NAM','NER','NGA','RWA','STP','SEN','SYC','SLE','ZAF','SSD',
            'TGO','UGA','TZA','ZMB','ZWE'],
    'AMR': ['ATG','ARG','BHS','BRB','BLZ','BOL','BRA','CAN','CHL','COL','CRI','CUB','DMA','DOM',
            'ECU','SLV','GRD','GTM','GUY','HTI','HND','JAM','MEX','NIC','PAN','PRY','PER','KNA',
            'LCA','VCT','SUR','TTO','USA','URY','VEN','ABW','AIA','CYM'],
    'EMR': ['AFG','BHR','DJI','EGY','IRN','IRQ','JOR','KWT','LBN','LBY','MAR','OMN','PAK','PSE',
            'QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM'],
    'EUR': ['ALB','AND','ARM','AUT','AZE','BLR','BEL','BIH','BGR','HRV','CYP','CZE','DNK','EST',
            'FIN','FRA','GEO','DEU','GRC','HUN','ISL','IRL','ISR','ITA','KAZ','KGZ','LVA','LTU',
            'LUX','MLT','MCO','MNE','NLD','MKD','NOR','POL','PRT','MDA','ROU','RUS','SMR','SRB',
            'SVK','SVN','ESP','SWE','CHE','TJK','TUR','TKM','UKR','GBR','UZB'],
    'SEAR': ['BGD','BTN','PRK','IND','IDN','MDV','MMR','NPL','LKA','THA','TLS'],
    'WPR':  ['AUS','BRN','KHM','CHN','COK','FJI','JPN','KIR','LAO','MYS','MHL','FSM','MNG','NRU',
             'NZL','NIU','PLW','PNG','PHL','KOR','WSM','SGP','SLB','TON','TUV','VUT','VNM','HKG','MAC','NCL'],
}
ISO3_TO_REGION = {iso: reg for reg, isos in WHO_REGIONS.items() for iso in isos}
REGION_ENC = {'AFR':0,'AMR':1,'EMR':2,'EUR':3,'SEAR':4,'WPR':5,'UNK':6}

era5_2022 = pd.read_csv(WEATHER_DIR / 'era5_weekly_2022_final.csv')

flu_train_iso3 = set(pd.read_csv(PROCESSED / 'features_flu_labeled.csv')['iso3'].unique())
df = flu_2022[flu_2022['iso3'].isin(flu_train_iso3)].copy()
df = df.merge(era5_2022[['iso3','iso_week','temp_c','solar_wm2','dewpoint_c','humidity_pct']],
              on=['iso3','iso_week'], how='inner')
df = df.sort_values(['iso3','iso_week']).reset_index(drop=True)

def lag_col(df, col, n, name):
    df[name] = df.groupby('iso3')[col].shift(n)
    return df

df = lag_col(df, 'temp_c',      4, 'temp_c_lag4w')
df = lag_col(df, 'solar_wm2',   8, 'solar_wm2_lag8w')
df = lag_col(df, 'dewpoint_c',  2, 'dewpoint_c_lag2w')
df = lag_col(df, 'humidity_pct',8, 'humidity_pct_lag8w')
df = lag_col(df, 'inf_cases',   1, 'inf_cases_lag1w')
df = lag_col(df, 'inf_cases',   2, 'inf_cases_lag2w')
df = lag_col(df, 'inf_cases',   4, 'inf_cases_lag4w')

# Drop rows where any lag is NaN (weeks 1-8 for lag8w)
lag_cols = ['temp_c_lag4w','solar_wm2_lag8w','dewpoint_c_lag2w','humidity_pct_lag8w',
            'inf_cases_lag1w','inf_cases_lag2w','inf_cases_lag4w']
df = df.dropna(subset=lag_cols).copy()

# log1p AR
for c in ['inf_cases_lag1w','inf_cases_lag2w','inf_cases_lag4w']:
    df[c] = np.log1p(df[c])

df['week_sin'] = np.sin(2 * np.pi * df['iso_week'] / 52)
df['week_cos'] = np.cos(2 * np.pi * df['iso_week'] / 52)
df['who_region_enc'] = df['iso3'].map(ISO3_TO_REGION).fillna('UNK').map(REGION_ENC)
df['quarter'] = ((df['iso_week'] - 1) // 13 + 1).clip(1, 4)

# Endemic channel labels using 2015-2019 frozen baseline (COVID-free)
flu_hist = pd.read_csv(PROCESSED / 'features_flu_labeled.csv',
                       usecols=['iso3','iso_week','ec_baseline','ec_upper'])
thr = flu_hist.groupby(['iso3','iso_week'])[['ec_baseline','ec_upper']].mean().reset_index()

flu_test = df.merge(thr, on=['iso3','iso_week'], how='left').dropna(subset=['ec_baseline'])

def risk_label(cases, base, upper):
    if base == 0 and upper == 0:
        return 'Low' if cases == 0 else 'High'
    return 'High' if cases >= upper else ('Medium' if cases >= base else 'Low')

flu_test['risk_label'] = flu_test.apply(
    lambda r: risk_label(r['inf_cases'], r['ec_baseline'], r['ec_upper']), axis=1)

print(f"Flu 2022 test: {len(flu_test):,} rows | {flu_test['iso3'].nunique()} countries "
      f"| weeks {flu_test['iso_week'].min()}-{flu_test['iso_week'].max()}")
print(f"Label dist: {flu_test['risk_label'].value_counts(normalize=True).round(3).to_dict()}")
print(f"AR max check (log1p): {flu_test['inf_cases_lag1w'].max():.2f}")
print(f"who_region_enc dist: {flu_test['who_region_enc'].value_counts().sort_index().to_dict()}")


In [ ]:
# [12.3] Dengue 2022: OpenDengue + ERA5 + lags + endemic labels

dengue_raw = pd.read_csv(RAW / 'National_extract_V1_3.csv')
d2022 = dengue_raw[(dengue_raw['Year']==2022) & (dengue_raw['T_res']=='Week')].copy()
d2022['calendar_start_date'] = pd.to_datetime(d2022['calendar_start_date'], dayfirst=False)
d2022['iso_week'] = d2022['calendar_start_date'].dt.isocalendar().week.astype(int)
d2022 = d2022.rename(columns={'ISO_A0':'iso3','Year':'iso_year'})
d2022 = (d2022.groupby(['iso3','iso_year','iso_week'], as_index=False)
               .agg({'dengue_total':'sum'}))

dengue_train_iso3 = set(pd.read_csv(PROCESSED / 'features_dengue_labeled.csv')['iso3'].unique())
d2022 = d2022[d2022['iso3'].isin(dengue_train_iso3)].copy()

era5_2022 = pd.read_csv(WEATHER_DIR / 'era5_weekly_2022_final.csv')
df = d2022.merge(era5_2022[['iso3','iso_week','temp_c','dewpoint_c','humidity_pct','precip_mm']],
                 on=['iso3','iso_week'], how='inner')
df = df.rename(columns={'temp_c':'temp_c_den'})
df = df.sort_values(['iso3','iso_week']).reset_index(drop=True)

df = lag_col(df, 'dewpoint_c',   2, 'dewpoint_c_den_lag2w')
df = lag_col(df, 'humidity_pct', 2, 'humidity_pct_den_lag2w')
df = lag_col(df, 'precip_mm',    2, 'precip_mm_den_lag2w')
df = lag_col(df, 'dengue_total', 1, 'dengue_total_lag1w')
df = lag_col(df, 'dengue_total', 2, 'dengue_total_lag2w')
df = lag_col(df, 'dengue_total', 4, 'dengue_total_lag4w')

df = df.dropna(subset=['dewpoint_c_den_lag2w','dengue_total_lag4w']).copy()

for c in ['dengue_total_lag1w','dengue_total_lag2w','dengue_total_lag4w']:
    df[c] = np.log1p(df[c])

df['week_sin'] = np.sin(2 * np.pi * df['iso_week'] / 52)
df['week_cos'] = np.cos(2 * np.pi * df['iso_week'] / 52)
df['who_region_enc'] = df['iso3'].map(ISO3_TO_REGION).fillna('UNK').map(REGION_ENC)
df['quarter'] = ((df['iso_week'] - 1) // 13 + 1).clip(1, 4)

dengue_hist = pd.read_csv(PROCESSED / 'features_dengue_labeled.csv',
                          usecols=['iso3','iso_week','ec_baseline','ec_upper'])
thr_d = dengue_hist.groupby(['iso3','iso_week'])[['ec_baseline','ec_upper']].mean().reset_index()

dengue_test = df.merge(thr_d, on=['iso3','iso_week'], how='left').dropna(subset=['ec_baseline'])
dengue_test['risk_label'] = dengue_test.apply(
    lambda r: risk_label(r['dengue_total'], r['ec_baseline'], r['ec_upper']), axis=1)

print(f"Dengue 2022 test: {len(dengue_test):,} rows | {dengue_test['iso3'].nunique()} countries "
      f"| weeks {dengue_test['iso_week'].min()}-{dengue_test['iso_week'].max()}")
print(f"Label dist: {dengue_test['risk_label'].value_counts(normalize=True).round(3).to_dict()}")


In [ ]:
# [12.4] Evaluate XGBoost champions on 2022 test set
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

LABEL_MAP = {'Low':0,'Medium':1,'High':2}

def evaluate_2022(test_df, pkl_path, disease):
    with open(pkl_path, 'rb') as fp:
        champ = pickle.load(fp)
    model    = champ['model']
    features = champ['features']

    mask   = test_df['risk_label'].map(LABEL_MAP).notna()
    X      = test_df.loc[mask, features]
    y_true = test_df.loc[mask, 'risk_label'].map(LABEL_MAP).astype(int)

    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)

    macro_f1 = f1_score(y_true, y_pred, average='macro')
    f1s      = f1_score(y_true, y_pred, average=None, labels=[0,1,2])
    try:
        auc = roc_auc_score(pd.get_dummies(y_true).values, y_proba,
                            multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')

    print(f"\n{'='*60}")
    print(f"{disease.upper()} -- 2022 TEST SET")
    print(f"{'='*60}")
    print(f"Rows: {len(X):,}  |  Countries: {test_df.loc[mask,'iso3'].nunique()}")
    print(f"macroF1={macro_f1:.4f}  AUC={auc:.4f}")
    print(f"F1(Low)={f1s[0]:.3f}  F1(Med)={f1s[1]:.3f}  F1(High)={f1s[2]:.3f}")
    print()
    print(classification_report(y_true, y_pred, target_names=['Low','Medium','High']))
    cm = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(cm, index=['True_Low','True_Med','True_High'],
                       columns=['Pred_Low','Pred_Med','Pred_High']))
    return {'macro_f1': macro_f1, 'auc': auc,
            'f1_low': f1s[0], 'f1_med': f1s[1], 'f1_high': f1s[2]}

flu_res    = evaluate_2022(flu_test,    PROCESSED / 'champion_flu_xgboost_tuned.pkl',    'flu')
dengue_res = evaluate_2022(dengue_test, PROCESSED / 'champion_dengue_xgboost_tuned.pkl', 'dengue')

print("\n" + "="*50)
print("SUMMARY: CV val (2017-2019) vs Test 2022")
print("="*50)
print(f"Flu:    CV macroF1=0.4876 | Test2022={flu_res['macro_f1']:.4f}")
print(f"Dengue: CV macroF1=0.5061 | Test2022={dengue_res['macro_f1']:.4f}")
print()
print("Note: 2022 labels dung 2015-2019 frozen baseline (COVID-free)")
print("COVID rebound effect (flu) se lam phan lon 2022 = High label -> model can overpredict High")


In [ ]:
# [12.5] Save 2022 test sets to CSV (cho RESTART nhanh o SESSION 13)
flu_test.to_csv(PROCESSED / 'flu_test_2022.csv',    index=False)
dengue_test.to_csv(PROCESSED / 'dengue_test_2022.csv', index=False)
print(f'Saved flu_test_2022.csv    ({len(flu_test):,} rows)')
print(f'Saved dengue_test_2022.csv ({len(dengue_test):,} rows)')


---

# SESSION 13 — Model Interpretability (SHAP)
> Giải thích từng dự đoán bằng SHAP TreeExplainer.
> Global: beeswarm feature importance. Local: waterfall cho 1 prediction cụ thể.
---

In [ ]:
# [13.0] RESTART SESSION 13 — load models + test data (khong can chay lai SESSION 12)

import pickle, pandas as pd, numpy as np
from pathlib import Path

# PROCESSED path (chay sau [0.1] + [0.4])
try:
    _ = PROCESSED
except NameError:
    BASE      = Path('/content/drive/MyDrive/KLTN')
    PROCESSED = BASE / 'dataset' / 'processed'

CLASS_NAMES = ['Low', 'Medium', 'High']
LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}

# Load champion models
_flu_champ    = pickle.load(open(PROCESSED / 'champion_flu_xgboost_tuned.pkl',    'rb'))
_dengue_champ = pickle.load(open(PROCESSED / 'champion_dengue_xgboost_tuned.pkl', 'rb'))
champ_flu    = _flu_champ['model']
champ_dengue = _dengue_champ['model']
FLU_FEATURES    = _flu_champ['features']
DENGUE_FEATURES = _dengue_champ['features']
print('Loaded champion models')
print(f'  Flu features:    {FLU_FEATURES}')
print(f'  Dengue features: {DENGUE_FEATURES}')

# Load 2022 test sets
flu_test    = pd.read_csv(PROCESSED / 'flu_test_2022.csv')
dengue_test = pd.read_csv(PROCESSED / 'dengue_test_2022.csv')
print(f'Loaded flu_test_2022.csv:    {flu_test.shape}')
print(f'Loaded dengue_test_2022.csv: {dengue_test.shape}')

# Prepare X, filtered DataFrames
mask_f = flu_test['risk_label'].notna()
mask_d = dengue_test['risk_label'].notna()
X_flu_test   = flu_test.loc[mask_f, FLU_FEATURES].reset_index(drop=True)
X_den_test   = dengue_test.loc[mask_d, DENGUE_FEATURES].reset_index(drop=True)
flu_test2    = flu_test.loc[mask_f].reset_index(drop=True)
dengue_test2 = dengue_test.loc[mask_d].reset_index(drop=True)
print(f'X_flu_test:   {X_flu_test.shape}')
print(f'X_den_test:   {X_den_test.shape}')
print('Ready. Co the chay [13.3] truc tiep (khong can SHAP). Chay [13.1] neu can SHAP.')


In [ ]:
# [13.1] SHAP global feature importance — beeswarm plot

try:
    import shap
    print(f'shap {shap.__version__}')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print(f'shap {shap.__version__} installed')

import pickle, matplotlib.pyplot as plt, pandas as pd, numpy as np

CLASS_NAMES = ['Low', 'Medium', 'High']
LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}

# Load champion models (pkl chua dict {model, features})
_flu_champ    = pickle.load(open(PROCESSED / 'champion_flu_xgboost_tuned.pkl',    'rb'))
_dengue_champ = pickle.load(open(PROCESSED / 'champion_dengue_xgboost_tuned.pkl', 'rb'))
champ_flu    = _flu_champ['model']
champ_dengue = _dengue_champ['model']
FLU_FEATURES    = _flu_champ['features']
DENGUE_FEATURES = _dengue_champ['features']
print('Loaded champion models')
print(f'Flu features ({len(FLU_FEATURES)}):    {FLU_FEATURES}')
print(f'Dengue features ({len(DENGUE_FEATURES)}): {DENGUE_FEATURES}')

# Test data tu SESSION 12
try:
    _ = flu_test
except NameError:
    raise RuntimeError('Chay SESSION 12 truoc de co flu_test va dengue_test')

# Debug: check columns available
print(f'\nflu_test cols:    {list(flu_test.columns)}')
print(f'dengue_test cols: {list(dengue_test.columns)}')

# Check features missing
missing_f = [c for c in FLU_FEATURES    if c not in flu_test.columns]
missing_d = [c for c in DENGUE_FEATURES if c not in dengue_test.columns]
if missing_f: print(f'MISSING flu features: {missing_f}')
if missing_d: print(f'MISSING dengue features: {missing_d}')

mask_f = flu_test['risk_label'].notna()
mask_d = dengue_test['risk_label'].notna()
X_flu_test   = flu_test.loc[mask_f, FLU_FEATURES].reset_index(drop=True)
X_den_test   = dengue_test.loc[mask_d, DENGUE_FEATURES].reset_index(drop=True)
flu_test2    = flu_test.loc[mask_f].reset_index(drop=True)
dengue_test2 = dengue_test.loc[mask_d].reset_index(drop=True)
print(f'\nflu test:    {X_flu_test.shape}')
print(f'dengue test: {X_den_test.shape}')

# SHAP TreeExplainer
print('\nComputing SHAP values (may take ~1 min)...')
explainer_flu    = shap.TreeExplainer(champ_flu)
explainer_dengue = shap.TreeExplainer(champ_dengue)
shap_flu    = explainer_flu(X_flu_test)    # shape: (n, 11, 3)
shap_dengue = explainer_dengue(X_den_test) # shape: (n, 11, 3)
print('Done.')

# Beeswarm: global importance for class HIGH
for name, sv, fnames in [
    ('FLU',    shap_flu,    FLU_FEATURES),
    ('DENGUE', shap_dengue, DENGUE_FEATURES)
]:
    plt.figure(figsize=(8, 5))
    shap.plots.beeswarm(sv[:, :, 2], show=False, max_display=11)
    plt.title(f'{name} — SHAP feature importance (class: High)', fontsize=12)
    plt.tight_layout()
    out = PROCESSED / f'shap_{name.lower()}_beeswarm_high.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')

# Bar: mean |SHAP| per feature (all 3 classes)
for name, sv, fnames in [
    ('FLU',    shap_flu,    FLU_FEATURES),
    ('DENGUE', shap_dengue, DENGUE_FEATURES)
]:
    mean_abs = np.abs(sv.values).mean(axis=(0, 2))
    order = np.argsort(mean_abs)[::-1]
    plt.figure(figsize=(7, 4))
    plt.barh([fnames[i] for i in order[::-1]], mean_abs[order[::-1]], color='steelblue')
    plt.xlabel('mean |SHAP value|')
    plt.title(f'{name} — Global feature importance (mean |SHAP|)', fontsize=11)
    plt.tight_layout()
    out = PROCESSED / f'shap_{name.lower()}_bar.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')


📌 **[13.1]** SHAP TreeExplainer cho XGBoost champion. Beeswarm plot: mỗi dot = 1 sample, trục X = SHAP value cho class High (dương = push toward High). Feature màu đỏ = giá trị cao, xanh = thấp. Bar chart = mean |SHAP| tổng hợp 3 class — đo lường feature nào model dựa vào nhiều nhất. Kỳ vọng: AR lags (inf_cases_lag1w) và temperature lag đứng đầu, xác nhận logic dịch tễ học.

In [ ]:
# [13.2] SHAP waterfall — giai thich 1 du doan cu the

def explain_prediction(df, X_test, model, shap_vals, features, disease_name):
    y_pred  = model.predict(X_test)
    y_true  = df['risk_label'].map(LABEL_MAP).values

    # Chon sample: predict dung High voi P(High) cao nhat
    probas  = model.predict_proba(X_test)
    mask_tp = (y_pred == 2) & (y_true == 2)
    if mask_tp.sum() == 0:
        print(f'[{disease_name}] Khong co True Positive High — dung sample predict High')
        mask_tp = (y_pred == 2)
    idx = np.where(mask_tp)[0][np.argmax(probas[mask_tp, 2])]

    row = df.iloc[idx]
    p   = probas[idx]
    print(f'\n=== {disease_name} ===')
    print(f"Country  : {row['iso3']}  Week {int(row['iso_week']):02d}/2022")
    print(f'Actual   : {CLASS_NAMES[int(y_true[idx])]}')
    print(f'Predicted: {CLASS_NAMES[y_pred[idx]]}  (P_Low={p[0]:.3f}, P_Med={p[1]:.3f}, P_High={p[2]:.3f})')

    print('\nFeature values:')
    for feat, val in zip(features, X_test.iloc[idx]):
        print(f'  {feat:25s} = {val:.4f}')

    # Waterfall — class High
    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(shap_vals[idx, :, 2], show=False, max_display=11)
    plt.title(
        f"{disease_name} SHAP Waterfall: {row['iso3']} w{int(row['iso_week']):02d}/2022 "
        f"-> {CLASS_NAMES[y_pred[idx]]} (P={p[2]:.2f})",
        fontsize=11
    )
    plt.tight_layout()
    fname = PROCESSED / f"shap_{disease_name.lower()}_waterfall_{row['iso3']}_w{int(row['iso_week']):02d}.png"
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname.name}')
    return idx

idx_flu    = explain_prediction(flu_test2,    X_flu_test, champ_flu,    shap_flu,    FLU_FEATURES,    'FLU')
idx_dengue = explain_prediction(dengue_test2, X_den_test, champ_dengue, shap_dengue, DENGUE_FEATURES, 'DENGUE')

# Force plot HTML
print('\nForce plots (HTML):')
for name, sv, X, idx in [
    ('flu',    shap_flu,    X_flu_test, idx_flu),
    ('dengue', shap_dengue, X_den_test, idx_dengue)
]:
    html = shap.force_plot(
        sv.base_values[idx, 2],
        sv.values[idx, :, 2],
        X.iloc[idx],
        matplotlib=False
    )
    shap.save_html(str(PROCESSED / f'shap_{name}_force.html'), html)
    print(f'  Saved: shap_{name}_force.html')


In [ ]:
# [13.3] Precision-Recall curve — justify threshold tuning for class High

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def analyze_threshold(model, X_test, y_true_series, disease_name):
    y_true   = y_true_series.map(LABEL_MAP).values
    y_proba  = model.predict_proba(X_test)
    p_high   = y_proba[:, 2]          # P(High) scores
    y_bin    = (y_true == 2).astype(int)

    precision, recall, thresholds = precision_recall_curve(y_bin, p_high)
    ap = average_precision_score(y_bin, p_high)

    # Current default operating point (argmax = threshold ~0.33 for balanced 3-class)
    default_pred = model.predict(X_test)
    default_prec = (y_bin[default_pred == 2]).mean() if (default_pred == 2).sum() > 0 else 0
    default_rec  = y_bin[default_pred == 2].sum() / y_bin.sum() if y_bin.sum() > 0 else 0
    default_f1   = (2 * default_prec * default_rec / (default_prec + default_rec)
                    if (default_prec + default_rec) > 0 else 0)

    # F1 for each threshold
    f1_scores = np.where(
        (precision[:-1] + recall[:-1]) > 0,
        2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1]),
        0
    )
    best_idx    = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_prec   = precision[best_idx]
    best_rec    = recall[best_idx]
    best_f1     = f1_scores[best_idx]

    # Threshold where recall >= 0.50
    rec50_idx   = np.where(recall[:-1] >= 0.50)[0]
    rec50_thresh = thresholds[rec50_idx[-1]] if len(rec50_idx) > 0 else None

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"{disease_name} — Class HIGH  (AP={ap:.4f})")
    print(f"{'='*55}")
    print(f"Prevalence High in test: {y_bin.mean()*100:.1f}%")
    print(f"\n  DEFAULT (argmax):  thresh≈auto  prec={default_prec:.3f}  rec={default_rec:.3f}  F1={default_f1:.3f}")
    print(f"  BEST F1 thresh:    thresh={best_thresh:.3f}  prec={best_prec:.3f}  rec={best_rec:.3f}  F1={best_f1:.3f}")
    if rec50_thresh is not None:
        r50_p = precision[rec50_idx[-1]]
        r50_r = recall[rec50_idx[-1]]
        r50_f = f1_scores[rec50_idx[-1]]
        print(f"  RECALL>=0.50:      thresh={rec50_thresh:.3f}  prec={r50_p:.3f}  rec={r50_r:.3f}  F1={r50_f:.3f}")
    else:
        print("  RECALL>=0.50: khong dat duoc voi bat ky threshold nao")

    # Threshold sensitivity table
    print(f"\n  Threshold sensitivity (High class):")
    print(f"  {'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
    for t in np.arange(0.20, 0.65, 0.05):
        mask = p_high >= t
        if mask.sum() == 0:
            continue
        p_ = y_bin[mask].mean()
        r_ = y_bin[mask].sum() / y_bin.sum() if y_bin.sum() > 0 else 0
        f_ = 2*p_*r_/(p_+r_) if (p_+r_) > 0 else 0
        marker = ' <-- default' if abs(t - 0.33) < 0.03 else ''
        marker = ' <-- best F1' if abs(t - best_thresh) < 0.03 else marker
        print(f"  {t:>10.2f}  {p_:>10.3f}  {r_:>8.3f}  {f_:>8.3f}{marker}")

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # PR curve
    axes[0].plot(recall, precision, color='steelblue', lw=2, label=f'AP={ap:.3f}')
    axes[0].scatter([default_rec], [default_prec], color='red',   s=100, zorder=5, label=f'Default (F1={default_f1:.3f})')
    axes[0].scatter([best_rec],    [best_prec],    color='green', s=100, zorder=5, label=f'Best F1 @ {best_thresh:.2f} (F1={best_f1:.3f})')
    axes[0].axhline(y_bin.mean(), ls='--', color='gray', alpha=0.5, label='Random baseline')
    axes[0].set_xlabel('Recall (High)'); axes[0].set_ylabel('Precision (High)')
    axes[0].set_title(f'{disease_name} — Precision-Recall curve (class High)')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    # F1 vs threshold
    axes[1].plot(thresholds, f1_scores, color='darkorange', lw=2)
    axes[1].axvline(best_thresh, ls='--', color='green', label=f'Best F1 thresh={best_thresh:.2f}')
    axes[1].axvline(1/3,         ls='--', color='red',   label='Default ~0.33')
    axes[1].set_xlabel('Threshold P(High)'); axes[1].set_ylabel('F1 (High class)')
    axes[1].set_title(f'{disease_name} — F1 vs Threshold')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out = PROCESSED / f'pr_curve_{disease_name.lower()}_high.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')

    return {'best_thresh': best_thresh, 'best_f1': best_f1,
            'default_f1': default_f1, 'ap': ap}

res_flu    = analyze_threshold(champ_flu,    X_flu_test, flu_test2['risk_label'],    'FLU')
res_dengue = analyze_threshold(champ_dengue, X_den_test, dengue_test2['risk_label'], 'DENGUE')

print(f"\n{'='*55}")
print("SUMMARY — Thu nghiem threshold tuning co dang khong?")
print(f"{'='*55}")
for name, res in [('FLU', res_flu), ('DENGUE', res_dengue)]:
    gain = res['best_f1'] - res['default_f1']
    verdict = 'NEN tune' if gain > 0.02 else 'Khong dang (cai thien < 0.02)'
    print(f"{name}: default F1={res['default_f1']:.3f} → best F1={res['best_f1']:.3f}  (+{gain:.3f})  → {verdict}")


📌 **[13.3]** Precision-Recall curve justify threshold tuning cho class High. Điểm đỏ = operating point hiện tại (argmax). Điểm xanh = threshold tối ưu F1. Nếu best F1 > default F1 + 0.02 → đáng tune. Sensitivity table cho thấy trade-off precision/recall tại từng threshold — giúp chọn threshold phù hợp với use case (cảnh báo sớm ưu tiên recall; tránh false alarm ưu tiên precision).

In [ ]:
# [13.4] Threshold tuning — evaluate macroF1 toan bo 3 class

from sklearn.metrics import f1_score, classification_report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def tune_threshold_multiclass(model, X_test, y_true_series, disease_name):
    """
    Threshold tuning dung cho multi-class:
      - predict High  neu P(High) > t
      - predict Low/Med neu P(High) <= t -> argmax(P(Low), P(Med))
    Evaluate macroF1 toan bo 3 class tai moi threshold.
    """
    y_true  = y_true_series.map(LABEL_MAP).values
    y_proba = model.predict_proba(X_test)
    p_high  = y_proba[:, 2]

    # Default prediction (argmax)
    y_default = model.predict(X_test)
    default_macro = f1_score(y_true, y_default, average='macro')
    default_f1s   = f1_score(y_true, y_default, average=None, labels=[0,1,2])

    print(f"\n{'='*60}")
    print(f"{disease_name} — Threshold tuning (macro-F1 toan 3 class)")
    print(f"{'='*60}")
    print(f"DEFAULT argmax: macroF1={default_macro:.4f}  "
          f"F1(Low)={default_f1s[0]:.3f}  F1(Med)={default_f1s[1]:.3f}  F1(High)={default_f1s[2]:.3f}")

    # Sweep threshold
    results = []
    for t in np.arange(0.10, 0.55, 0.01):
        # Multi-class prediction with threshold
        y_pred = np.where(
            p_high > t,
            2,                          # predict High
            np.argmax(y_proba[:, :2], axis=1)  # argmax of Low/Med only
        )
        macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f1s   = f1_score(y_true, y_pred, average=None, labels=[0,1,2], zero_division=0)
        results.append({
            'threshold': round(t, 2),
            'macroF1':   macro,
            'F1_Low':    f1s[0],
            'F1_Med':    f1s[1],
            'F1_High':   f1s[2],
        })

    df_res = pd.DataFrame(results)
    best_idx   = df_res['macroF1'].idxmax()
    best_row   = df_res.loc[best_idx]
    macro_gain = best_row['macroF1'] - default_macro

    # Print table (every 0.05 step for readability)
    print(f"\n  {'Thresh':>7}  {'macroF1':>9}  {'F1(Low)':>8}  {'F1(Med)':>8}  {'F1(High)':>9}")
    for _, r in df_res[df_res['threshold'].isin(np.arange(0.10, 0.55, 0.05).round(2))].iterrows():
        marker = ' <-- BEST' if r['threshold'] == best_row['threshold'] else ''
        marker = ' <-- default' if abs(r['threshold'] - 0.33) < 0.03 else marker
        print(f"  {r['threshold']:>7.2f}  {r['macroF1']:>9.4f}  "
              f"{r['F1_Low']:>8.3f}  {r['F1_Med']:>8.3f}  {r['F1_High']:>9.3f}{marker}")

    print(f"\n  BEST thresh={best_row['threshold']:.2f}: "
          f"macroF1={best_row['macroF1']:.4f}  "
          f"(+{macro_gain:.4f} vs default)")
    print(f"  F1(Low)={best_row['F1_Low']:.3f}  "
          f"F1(Med)={best_row['F1_Med']:.3f}  "
          f"F1(High)={best_row['F1_High']:.3f}")

    verdict = 'NEN tune' if macro_gain > 0.005 else 'Khong dang (macroF1 gain < 0.005)'
    print(f"\n  VERDICT: {verdict}")

    if macro_gain > 0.005:
        print(f"\n  Classification report tai best threshold={best_row['threshold']:.2f}:")
        y_best = np.where(p_high > best_row['threshold'], 2,
                          np.argmax(y_proba[:, :2], axis=1))
        print(classification_report(y_true, y_best,
                                    target_names=['Low','Medium','High'],
                                    zero_division=0))

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(df_res['threshold'], df_res['macroF1'],  color='steelblue', lw=2, label='macroF1')
    axes[0].plot(df_res['threshold'], df_res['F1_High'],  color='red',       lw=1.5, ls='--', label='F1(High)')
    axes[0].plot(df_res['threshold'], df_res['F1_Low'],   color='green',     lw=1.5, ls='--', label='F1(Low)')
    axes[0].plot(df_res['threshold'], df_res['F1_Med'],   color='orange',    lw=1.5, ls='--', label='F1(Med)')
    axes[0].axvline(best_row['threshold'], ls=':', color='steelblue', label=f"Best macroF1 thresh={best_row['threshold']:.2f}")
    axes[0].axvline(1/3, ls=':', color='gray', label='Default ~0.33')
    axes[0].axhline(default_macro, ls='--', color='gray', alpha=0.5, label=f'Default macroF1={default_macro:.3f}')
    axes[0].set_xlabel('Threshold P(High)')
    axes[0].set_ylabel('F1 score')
    axes[0].set_title(f'{disease_name} — macroF1 vs Threshold')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_res['threshold'], df_res['macroF1'] - default_macro,
                 color='darkgreen', lw=2)
    axes[1].axhline(0,    ls='--', color='gray')
    axes[1].axhline(0.01, ls=':', color='orange', label='+0.01 gain line')
    axes[1].axvline(best_row['threshold'], ls=':', color='steelblue',
                    label=f"Best thresh={best_row['threshold']:.2f}")
    axes[1].set_xlabel('Threshold P(High)')
    axes[1].set_ylabel('macroF1 gain vs default')
    axes[1].set_title(f'{disease_name} — macroF1 gain from threshold tuning')
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    out = PROCESSED / f'threshold_tune_{disease_name.lower()}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')

    return {
        'best_thresh': best_row['threshold'],
        'best_macro':  best_row['macroF1'],
        'default_macro': default_macro,
        'gain': macro_gain,
        'df': df_res
    }

res_flu    = tune_threshold_multiclass(champ_flu,    X_flu_test, flu_test2['risk_label'],    'FLU')
res_dengue = tune_threshold_multiclass(champ_dengue, X_den_test, dengue_test2['risk_label'], 'DENGUE')

print(f"\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")
for name, res in [('FLU', res_flu), ('DENGUE', res_dengue)]:
    print(f"{name:8s}: default macroF1={res['default_macro']:.4f} "
          f"-> best macroF1={res['best_macro']:.4f} "
          f"(+{res['gain']:.4f}) "
          f"at thresh={res['best_thresh']:.2f}  "
          f"-> {'NEN tune' if res['gain'] > 0.005 else 'Khong dang'}")


📌 **[13.4]** Threshold tuning đúng cho multi-class: predict High nếu P(High) > t, ngược lại argmax(P(Low), P(Med)). Metric quyết định là **macroF1 toàn 3 class**, không phải F1(High) đơn lẻ. Nếu best macroF1 > default + 0.005 → đáng tune. Plot trái: thấy F1 các class thay đổi theo threshold. Plot phải: macroF1 gain — dương = tốt hơn default, âm = xấu hơn.

In [ ]:
# [13.5] Save threshold vao champion pkl + final metrics

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Threshold da chon tu [13.4]
THRESHOLD_FLU    = 0.32   # justified: macroF1 +0.034
THRESHOLD_DENGUE = None   # argmax (khong tune)

# ── Update champion pkl ───────────────────────────────────────────────────────
for disease, threshold in [('flu', THRESHOLD_FLU), ('dengue', THRESHOLD_DENGUE)]:
    pkl_path = PROCESSED / f'champion_{disease}_xgboost_tuned.pkl'
    champ = pickle.load(open(pkl_path, 'rb'))
    champ['threshold'] = threshold
    with open(pkl_path, 'wb') as f:
        pickle.dump(champ, f)
    thresh_str = f'{threshold:.2f}' if threshold is not None else 'argmax (None)'
    print(f'Updated {pkl_path.name}  threshold={thresh_str}')

# ── Helper: predict with threshold ───────────────────────────────────────────
def predict_with_threshold(model, X, threshold=None):
    y_proba = model.predict_proba(X)
    if threshold is None:
        return model.predict(X), y_proba
    y_pred = np.where(
        y_proba[:, 2] > threshold,
        2,
        np.argmax(y_proba[:, :2], axis=1)
    )
    return y_pred, y_proba

# ── Final evaluation ──────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("FINAL METRICS — XGBoost Champion (2022 test set)")
print(f"{'='*65}")

results = []
for disease, model, X_test, test_df, threshold in [
    ('FLU',    champ_flu,    X_flu_test, flu_test2,    THRESHOLD_FLU),
    ('DENGUE', champ_dengue, X_den_test, dengue_test2, THRESHOLD_DENGUE),
]:
    y_true  = test_df['risk_label'].map(LABEL_MAP).values
    y_pred, y_proba = predict_with_threshold(model, X_test, threshold)

    macro_f1 = f1_score(y_true, y_pred, average='macro')
    f1s      = f1_score(y_true, y_pred, average=None, labels=[0,1,2])
    try:
        auc = roc_auc_score(pd.get_dummies(y_true).values, y_proba,
                            multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')

    thresh_str = f'{threshold:.2f}' if threshold is not None else 'argmax'
    print(f"\n{disease} (threshold={thresh_str})")
    print(f"  macroF1={macro_f1:.4f}  AUC={auc:.4f}")
    print(f"  F1(Low)={f1s[0]:.3f}  F1(Med)={f1s[1]:.3f}  F1(High)={f1s[2]:.3f}")
    print()
    print(classification_report(y_true, y_pred,
                                target_names=['Low','Medium','High'],
                                zero_division=0))
    results.append({
        'disease': disease, 'threshold': thresh_str,
        'macroF1': macro_f1, 'AUC': auc,
        'F1_Low': f1s[0], 'F1_Med': f1s[1], 'F1_High': f1s[2]
    })

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("IMPROVEMENT SUMMARY — Before vs After threshold tuning")
print(f"{'='*65}")
before = {
    'FLU':    {'macroF1': 0.5090, 'F1_High': 0.487},
    'DENGUE': {'macroF1': 0.5441, 'F1_High': 0.446},
}
for r in results:
    b = before[r['disease']]
    dm = r['macroF1'] - b['macroF1']
    dh = r['F1_High'] - b['F1_High']
    print(f"{r['disease']:8s}: macroF1 {b['macroF1']:.4f} -> {r['macroF1']:.4f} ({dm:+.4f}) | "
          f"F1(High) {b['F1_High']:.3f} -> {r['F1_High']:.3f} ({dh:+.3f})")

print("\nThreshold config saved to champion pkl files.")
print("Inference: dung predict_with_threshold(model, X, champ['threshold'])")


📌 **[13.5]** Lưu `threshold` vào champion pkl — đảm bảo inference pipeline dùng đúng threshold đã tune, không hardcode rải rác. FLU threshold=0.32 (macroF1 +0.034, F1(High) +0.120). DENGUE threshold=None (argmax — gain chỉ +0.003, không đáng). Helper `predict_with_threshold()` sẽ được tái sử dụng trong backend FastAPI.

📌 **[13.2]** Waterfall plot giải thích 1 dự đoán cụ thể: trục X = SHAP contribution cho class High, bar đỏ = feature đẩy prediction về phía High, bar xanh = kéo về Low/Med. E[f(x)] = base value (prior trung bình), f(x) = final prediction score. Tổng tất cả SHAP contributions + base value = log-odds của P(High). Ví dụ: inf_cases_lag1w cao + nhiệt độ thấp → model tự tin predict High. Force plot HTML lưu để embed vào dashboard nếu cần.